<a href="https://colab.research.google.com/github/lizlorenzana931109-art/FinPredict-Actividad6/blob/main/nootebook/00_exploracion_dataset_actividad6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [35]:
# ============================================================
# FINPREDICT | EXPLORACIÓN PREVIA A ACTIVIDAD 6
# 01. CONFIGURACIÓN DEL ENTORNO
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

# Reproducibilidad
SEED = 2026

# Ruta del repositorio de Actividad 5
RUTA_REPO = Path("/content/FinPredict-Actividad5")

# Rutas principales
RUTA_DATOS_INI = RUTA_REPO / "datos" / "datos_ini"
RUTA_DATOS_LIMP = RUTA_REPO / "datos" / "datos_limp"

print("Configuración cargada correctamente.")
print(f"Semilla de reproducibilidad: {SEED}")
print(f"Repositorio: {RUTA_REPO}")

Configuración cargada correctamente.
Semilla de reproducibilidad: 2026
Repositorio: /content/FinPredict-Actividad5


In [36]:
# ============================================================
# FINPREDICT | EXPLORACIÓN PREVIA A ACTIVIDAD 6
# 02. CARGA CONTROLADA DE DATASETS V1
# ============================================================

# Fuentes originales
df_contratos = pd.read_parquet(
    RUTA_DATOS_INI / "contratos.parquet"
)

df_obligaciones = pd.read_parquet(
    RUTA_DATOS_INI / "obligaciones.parquet"
)

# Datasets analíticos
df_analitico_v0 = pd.read_parquet(
    RUTA_DATOS_INI / "finpredict_analitico_v0.parquet"
)

df_preparado_v1 = pd.read_parquet(
    RUTA_DATOS_LIMP / "finpredict_preparado_v1.parquet"
)

df_desarrollo_v1 = pd.read_parquet(
    RUTA_DATOS_LIMP / "finpredict_desarrollo_v1.parquet"
)

df_evaluacion_v1 = pd.read_parquet(
    RUTA_DATOS_LIMP / "finpredict_evaluacion_v1.parquet"
)

# Resumen estructural
resumen_carga = pd.DataFrame({
    "dataset": [
        "contratos",
        "obligaciones",
        "analitico_v0",
        "preparado_v1",
        "desarrollo_v1",
        "evaluacion_v1"
    ],
    "filas": [
        len(df_contratos),
        len(df_obligaciones),
        len(df_analitico_v0),
        len(df_preparado_v1),
        len(df_desarrollo_v1),
        len(df_evaluacion_v1)
    ],
    "columnas": [
        df_contratos.shape[1],
        df_obligaciones.shape[1],
        df_analitico_v0.shape[1],
        df_preparado_v1.shape[1],
        df_desarrollo_v1.shape[1],
        df_evaluacion_v1.shape[1]
    ]
})

display(resumen_carga)

print("Datasets V1 cargados correctamente.")

,dataset,filas,columnas
0,contratos,50000,10
1,obligaciones,1475182,4
2,analitico_v0,390661,30
3,preparado_v1,390661,34
4,desarrollo_v1,350520,34
5,evaluacion_v1,40141,34


Datasets V1 cargados correctamente.


In [37]:
# ============================================================
# FINPREDICT | EXPLORACIÓN PREVIA A ACTIVIDAD 6
# 03. DIAGNÓSTICO DEL DATASET V1
# ============================================================

# Copia de trabajo para evitar modificar el dataset original
df_diag = df_desarrollo_v1.copy()

# ------------------------------------------------------------
# 1. Calidad estructural
# ------------------------------------------------------------

resumen_calidad = pd.DataFrame({
    "indicador": [
        "Registros",
        "Columnas",
        "Créditos únicos",
        "Valores nulos",
        "Duplicados",
        "Incumplimientos",
        "Prevalencia positiva (%)"
    ],
    "valor": [
        len(df_diag),
        df_diag.shape[1],
        df_diag["id_credito"].nunique(),
        int(df_diag.isna().sum().sum()),
        int(df_diag.duplicated().sum()),
        int(df_diag["incumplimiento_90d"].sum()),
        round(df_diag["incumplimiento_90d"].mean() * 100, 2)
    ]
})

print("DIAGNÓSTICO GENERAL V1")
display(resumen_calidad)

# ------------------------------------------------------------
# 2. Distribución del target
# ------------------------------------------------------------

distribucion_target = (
    df_diag["incumplimiento_90d"]
    .value_counts()
    .sort_index()
    .rename_axis("clase")
    .reset_index(name="registros")
)

distribucion_target["porcentaje"] = (
    distribucion_target["registros"]
    / len(df_diag)
    * 100
).round(2)

print("\nDISTRIBUCIÓN DEL TARGET")
display(distribucion_target)

# ------------------------------------------------------------
# 3. Riesgo por rango de días de atraso
# ------------------------------------------------------------

df_diag["rango_atraso"] = pd.cut(
    df_diag["dias_atraso"],
    bins=[-1, 0, 15, 30, 45, 60, 75, 90, float("inf")],
    labels=[
        "0 días",
        "1-15 días",
        "16-30 días",
        "31-45 días",
        "46-60 días",
        "61-75 días",
        "76-90 días",
        ">90 días"
    ]
)

riesgo_por_atraso_v1 = (
    df_diag
    .groupby("rango_atraso", observed=False)
    .agg(
        registros=("incumplimiento_90d", "size"),
        incumplimientos=("incumplimiento_90d", "sum"),
        tasa_incumplimiento=("incumplimiento_90d", "mean")
    )
    .reset_index()
)

riesgo_por_atraso_v1["tasa_incumplimiento_%"] = (
    riesgo_por_atraso_v1["tasa_incumplimiento"] * 100
).round(2)

print("\nRIESGO POR RANGO DE ATRASO")
display(
    riesgo_por_atraso_v1[
        [
            "rango_atraso",
            "registros",
            "incumplimientos",
            "tasa_incumplimiento_%"
        ]
    ]
)

# ------------------------------------------------------------
# 4. Variables con mayor diferencia entre clases
# ------------------------------------------------------------

variables_revision = [
    "saldo_vencido",
    "dias_atraso",
    "saldo_vencido_mensualidades",
    "importe_pagado_3m",
    "numero_pagos_3m",
    "porcentaje_pago_3m",
    "pagos_parciales_3m",
    "regularizaciones_3m",
    "relacion_pago_ingreso"
]

comparacion_clases = (
    df_diag
    .groupby("incumplimiento_90d")[variables_revision]
    .mean()
    .T
)

comparacion_clases.columns = [
    "media_clase_0",
    "media_clase_1"
]

comparacion_clases["diferencia"] = (
    comparacion_clases["media_clase_1"]
    - comparacion_clases["media_clase_0"]
)

print("\nCOMPARACIÓN DE VARIABLES ENTRE CLASES")
display(comparacion_clases)

DIAGNÓSTICO GENERAL V1


,indicador,valor
0,Registros,350520.0
1,Columnas,34.0
2,Créditos únicos,42492.0
3,Valores nulos,0.0
4,Duplicados,0.0
5,Incumplimientos,21368.0
6,Prevalencia positiva (%),6.1



DISTRIBUCIÓN DEL TARGET


,clase,registros,porcentaje
0,0,329152,93.9
1,1,21368,6.1



RIESGO POR RANGO DE ATRASO


,rango_atraso,registros,incumplimientos,tasa_incumplimiento_%
0,0 días,131257,24,0.02
1,1-15 días,88498,596,0.67
2,16-30 días,93846,924,0.98
3,31-45 días,12889,3510,27.23
4,46-60 días,11114,4422,39.79
5,61-75 días,6837,5853,85.61
6,76-90 días,6079,6039,99.34
7,>90 días,0,0,NaN



COMPARACIÓN DE VARIABLES ENTRE CLASES


,media_clase_0,media_clase_1,diferencia
saldo_vencido,5407.949405,57970.168848,52562.219443
dias_atraso,10.870929,60.659163,49.788234
saldo_vencido_mensualidades,0.188403,2.017894,1.829491
importe_pagado_3m,86644.780682,74030.251973,-12614.528709
numero_pagos_3m,3.361684,2.599261,-0.762423
porcentaje_pago_3m,1.008271,0.861897,-0.146373
pagos_parciales_3m,0.201761,0.757815,0.556055
regularizaciones_3m,0.451074,0.175636,-0.275438
relacion_pago_ingreso,0.325186,0.327466,0.002279


### Conclusión del diagnóstico V1

El dataset V1 presenta una estructura consistente, sin valores nulos ni registros
duplicados, y conserva un desbalance de clases coherente con un problema de riesgo
crediticio.

Sin embargo, el análisis evidencia una relación excesivamente fuerte entre los días
de atraso al corte y la variable objetivo `incumplimiento_90d`. En particular, las
observaciones con 61 a 75 días de atraso presentan una tasa de incumplimiento de
85.61%, mientras que aquellas con 76 a 90 días alcanzan 99.34%.

Este comportamiento puede facilitar una discriminación casi determinista del target
y producir métricas artificialmente elevadas durante la evaluación del modelo.

Por esta razón, se desarrollará una versión V2 del proceso sintético, conservando la
definición temporal del incumplimiento y el desbalance natural del problema, pero
introduciendo mayor heterogeneidad, recuperación y variabilidad en el comportamiento
de pago.

El objetivo no es reducir artificialmente el desempeño del modelo, sino construir un
problema de clasificación más representativo y adecuado para evaluar generalización,
ajuste de umbral, ROC-AUC, validación cruzada y comparación estadística de modelos.

In [38]:
# ============================================================
# FINPREDICT | EXPLORACIÓN PREVIA A ACTIVIDAD 6
# 04. MOTOR V2 COMPLETO
# ============================================================

# ------------------------------------------------------------
# 1. Distribución de perfiles
# ------------------------------------------------------------

DISTRIBUCION_PERFILES_V2 = {
    "REGULAR": 0.60,
    "RETRASO_OCASIONAL": 0.20,
    "IRREGULAR": 0.15,
    "DETERIORO_PROGRESIVO": 0.05
}

# ------------------------------------------------------------
# 2. Reglas base de pago
# ------------------------------------------------------------

REGLAS_PAGO_V2 = {
    "REGULAR": {
        "completo": 0.90,
        "parcial": 0.07,
        "sin_pago": 0.03,
        "regularizacion": 0.60
    },
    "RETRASO_OCASIONAL": {
        "completo": 0.78,
        "parcial": 0.15,
        "sin_pago": 0.07,
        "regularizacion": 0.45
    },
    "IRREGULAR": {
        "completo": 0.58,
        "parcial": 0.27,
        "sin_pago": 0.15,
        "regularizacion": 0.30
    },
    "DETERIORO_PROGRESIVO": {
        "completo": 0.68,
        "parcial": 0.22,
        "sin_pago": 0.10,
        "regularizacion": 0.20
    }
}

for perfil, reglas in REGLAS_PAGO_V2.items():
    suma = (
        reglas["completo"]
        + reglas["parcial"]
        + reglas["sin_pago"]
    )

    assert np.isclose(suma, 1.0)
    assert 0 <= reglas["regularizacion"] <= 1


# ------------------------------------------------------------
# 3. Asignación reproducible de perfiles
# ------------------------------------------------------------

def asignar_perfiles_v2(df_contratos, seed=SEED):

    n = len(df_contratos)

    conteos = {
        perfil: int(n * proporcion)
        for perfil, proporcion
        in DISTRIBUCION_PERFILES_V2.items()
    }

    # Ajuste defensivo por redondeos
    diferencia = n - sum(conteos.values())

    if diferencia != 0:
        conteos["REGULAR"] += diferencia

    perfiles = np.concatenate([
        np.repeat(perfil, cantidad)
        for perfil, cantidad in conteos.items()
    ])

    rng = np.random.default_rng(seed)
    rng.shuffle(perfiles)

    df_perfiles = pd.DataFrame({
        "id_credito": df_contratos["id_credito"].to_numpy(),
        "perfil_simulacion": perfiles
    })

    return df_perfiles


# ------------------------------------------------------------
# 4. Cálculo simple de saldo vencido a una fecha
# ------------------------------------------------------------

def calcular_saldo_vencido_v2(
    obligaciones_contrato,
    movimientos_contrato,
    fecha_evaluacion
):
    """
    Calcula deuda exigible menos pagos realizados hasta una fecha.
    """

    obligaciones = obligaciones_contrato.loc[
        obligaciones_contrato["fecha_vencimiento"] <= fecha_evaluacion
    ].copy()

    if obligaciones.empty:
        return 0.0

    total_exigible = obligaciones["importe_exigible"].sum()

    if movimientos_contrato.empty:
        total_pagado = 0.0
    else:
        total_pagado = movimientos_contrato.loc[
            movimientos_contrato["fecha_movimiento"] <= fecha_evaluacion,
            "importe"
        ].sum()

    return max(
        round(float(total_exigible - total_pagado), 2),
        0.0
    )


# ------------------------------------------------------------
# 5. Generación dinámica de movimientos ordinarios
# ------------------------------------------------------------

def generar_movimientos_contrato_v2(
    obligaciones_contrato,
    perfil,
    generador,
    fecha_fin
):

    reglas = REGLAS_PAGO_V2[perfil]

    movimientos = []

    obligaciones_ordenadas = (
        obligaciones_contrato
        .sort_values("fecha_vencimiento")
        .reset_index(drop=True)
    )

    total = len(obligaciones_ordenadas)

    # Heterogeneidad estable por contrato
    factor_individual = generador.normal(
        loc=0.0,
        scale=0.04
    )

    omisiones_consecutivas = 0
    pagos_completos_consecutivos = 0

    for posicion, obligacion in enumerate(
        obligaciones_ordenadas.itertuples(index=False)
    ):

        prob_completo = reglas["completo"]
        prob_parcial = reglas["parcial"]
        prob_sin_pago = reglas["sin_pago"]

        # Heterogeneidad individual
        prob_completo -= factor_individual
        prob_sin_pago += factor_individual

        # Memoria negativa
        if omisiones_consecutivas > 0:
            ajuste = min(
                0.04 * omisiones_consecutivas,
                0.12
            )

            prob_completo -= ajuste
            prob_sin_pago += ajuste

        # Memoria positiva
        if pagos_completos_consecutivos >= 2:
            ajuste = min(
                0.02 * pagos_completos_consecutivos,
                0.08
            )

            prob_completo += ajuste
            prob_sin_pago -= ajuste

        # Deterioro progresivo más suave
        if perfil == "DETERIORO_PROGRESIVO":
            avance = (
                posicion /
                max(total - 1, 1)
            )

            prob_completo -= 0.18 * avance
            prob_parcial += 0.10 * avance
            prob_sin_pago += 0.08 * avance

        # Shock temporal adverso
        if generador.random() < 0.035:
            prob_completo -= 0.20
            prob_parcial += 0.08
            prob_sin_pago += 0.12

        probabilidades = np.array([
            prob_completo,
            prob_parcial,
            prob_sin_pago
        ])

        probabilidades = np.clip(
            probabilidades,
            0.02,
            None
        )

        probabilidades = (
            probabilidades /
            probabilidades.sum()
        )

        resultado = generador.choice(
            ["COMPLETO", "PARCIAL", "SIN_PAGO"],
            p=probabilidades
        )

        if resultado == "SIN_PAGO":
            omisiones_consecutivas += 1
            pagos_completos_consecutivos = 0
            continue

        if resultado == "COMPLETO":
            pagos_completos_consecutivos += 1
            omisiones_consecutivas = max(
                omisiones_consecutivas - 1,
                0
            )

            importe = float(
                obligacion.importe_exigible
            )

            desfase = int(
                generador.integers(-3, 15)
            )

        else:
            pagos_completos_consecutivos = 0
            omisiones_consecutivas = max(
                omisiones_consecutivas - 1,
                0
            )

            factor_pago = generador.uniform(
                0.25,
                0.85
            )

            importe = (
                float(obligacion.importe_exigible)
                * factor_pago
            )

            desfase = int(
                generador.integers(0, 30)
            )

        fecha_pago = (
            obligacion.fecha_vencimiento
            + pd.Timedelta(days=desfase)
        )

        if fecha_pago > fecha_fin:
            continue

        movimientos.append({
            "id_credito": obligacion.id_credito,
            "numero_pago_origen": obligacion.numero_pago,
            "fecha_movimiento": fecha_pago,
            "tipo_movimiento": generador.choice(
                ["SPEI", "CAJA"],
                p=[0.75, 0.25]
            ),
            "importe": round(importe, 2),
            "resultado_simulado": resultado
        })

    columnas = [
        "id_credito",
        "numero_pago_origen",
        "fecha_movimiento",
        "tipo_movimiento",
        "importe",
        "resultado_simulado"
    ]

    return pd.DataFrame(
        movimientos,
        columns=columnas
    )


# ------------------------------------------------------------
# 6. Regularizaciones V2
# ------------------------------------------------------------

def generar_regularizaciones_v2(
    obligaciones_contrato,
    movimientos_contrato,
    perfil,
    generador,
    fecha_fin
):

    probabilidad_base = (
        REGLAS_PAGO_V2[perfil]["regularizacion"]
    )

    obligaciones = (
        obligaciones_contrato
        .sort_values("fecha_vencimiento")
        .copy()
    )

    nuevos_movimientos = []

    fechas_revision = pd.date_range(
        start=obligaciones["fecha_vencimiento"].min(),
        end=fecha_fin,
        freq="QE-DEC"
    )

    for fecha in fechas_revision:

        if nuevos_movimientos:
            historial = pd.concat(
                [
                    movimientos_contrato,
                    pd.DataFrame(nuevos_movimientos)
                ],
                ignore_index=True
            )
        else:
            historial = movimientos_contrato

        saldo_vencido = calcular_saldo_vencido_v2(
            obligaciones,
            historial,
            fecha
        )

        if saldo_vencido <= 0:
            continue

        probabilidad = np.clip(
            probabilidad_base
            + generador.normal(0, 0.05),
            0.05,
            0.90
        )

        if generador.random() >= probabilidad:
            continue

        factor = generador.uniform(
            0.25,
            0.80
        )

        importe_extra = round(
            saldo_vencido * factor,
            2
        )

        if importe_extra <= 0:
            continue

        nuevos_movimientos.append({
            "id_credito": obligaciones["id_credito"].iloc[0],
            "numero_pago_origen": pd.NA,
            "fecha_movimiento": fecha,
            "tipo_movimiento": generador.choice(
                ["SPEI", "CAJA"],
                p=[0.75, 0.25]
            ),
            "importe": importe_extra,
            "resultado_simulado": "REGULARIZACION"
        })

    columnas = [
        "id_credito",
        "numero_pago_origen",
        "fecha_movimiento",
        "tipo_movimiento",
        "importe",
        "resultado_simulado"
    ]

    return pd.DataFrame(
        nuevos_movimientos,
        columns=columnas
    )


# ------------------------------------------------------------
# 7. Motor integrado V2
# ------------------------------------------------------------

def simular_movimientos_finpredict_v2(
    obligaciones_contrato,
    perfil,
    generador,
    fecha_fin
):

    ordinarios = generar_movimientos_contrato_v2(
        obligaciones_contrato,
        perfil,
        generador,
        fecha_fin
    )

    regularizaciones = generar_regularizaciones_v2(
        obligaciones_contrato,
        ordinarios,
        perfil,
        generador,
        fecha_fin
    )

    partes = [
        df
        for df in [ordinarios, regularizaciones]
        if not df.empty
    ]

    columnas = [
        "id_credito",
        "numero_pago_origen",
        "fecha_movimiento",
        "tipo_movimiento",
        "importe",
        "resultado_simulado"
    ]

    if not partes:
        return pd.DataFrame(columns=columnas)

    movimientos = (
        pd.concat(
            partes,
            ignore_index=True
        )
        .sort_values(
            ["fecha_movimiento", "numero_pago_origen"],
            na_position="last",
            kind="stable"
        )
        .reset_index(drop=True)
    )

    return movimientos


print("Motor V2 cargado correctamente.")

Motor V2 cargado correctamente.


In [41]:
# ============================================================
# FINPREDICT | EXPLORACIÓN PREVIA A ACTIVIDAD 6
# 05. PILOTO V2 CON 5,000 CONTRATOS
# ============================================================

# ------------------------------------------------------------
# 1. Asignar perfiles V2
# ------------------------------------------------------------

df_perfiles_v2 = asignar_perfiles_v2(
    df_contratos,
    seed=SEED
)

# ------------------------------------------------------------
# 2. Seleccionar muestra piloto reproducible
# ------------------------------------------------------------

df_piloto_creditos = (
    df_perfiles_v2
    .sample(
        n=5000,
        random_state=SEED
    )
    .reset_index(drop=True)
)

print("Distribución de perfiles en el piloto:")

display(
    df_piloto_creditos[
        "perfil_simulacion"
    ]
    .value_counts()
    .rename_axis("perfil")
    .reset_index(name="creditos")
)

# ------------------------------------------------------------
# 3. Preparar fechas
# ------------------------------------------------------------

df_obligaciones["fecha_vencimiento"] = pd.to_datetime(
    df_obligaciones["fecha_vencimiento"],
    errors="coerce"
)

FECHA_FIN_V2 = df_obligaciones[
    "fecha_vencimiento"
].max()

print(f"\nFecha máxima de simulación: {FECHA_FIN_V2.date()}")

# ------------------------------------------------------------
# 4. Simular movimientos
# ------------------------------------------------------------

movimientos_piloto = []

creditos_sin_obligaciones = []
creditos_sin_fecha_valida = []

for fila in df_piloto_creditos.itertuples(index=False):

    id_credito = fila.id_credito
    perfil = fila.perfil_simulacion

    obligaciones_credito = (
        df_obligaciones.loc[
            df_obligaciones["id_credito"] == id_credito
        ]
        .sort_values("numero_pago")
        .copy()
    )

    # Control 1: contrato sin obligaciones
    if obligaciones_credito.empty:
        creditos_sin_obligaciones.append(id_credito)
        continue

    # Control 2: contrato sin fechas válidas
    obligaciones_credito = (
        obligaciones_credito
        .dropna(subset=["fecha_vencimiento"])
        .copy()
    )

    if obligaciones_credito.empty:
        creditos_sin_fecha_valida.append(id_credito)
        continue

    # Semilla independiente y reproducible por crédito
    numero_credito = int(
        id_credito.replace("HIP", "")
    )

    rng_credito = np.random.default_rng(
        SEED + numero_credito
    )

    movimientos_credito = (
        simular_movimientos_finpredict_v2(
            obligaciones_credito,
            perfil,
            rng_credito,
            FECHA_FIN_V2
        )
    )

    if not movimientos_credito.empty:
        movimientos_piloto.append(
            movimientos_credito
        )

# ------------------------------------------------------------
# 5. Consolidar resultados
# ------------------------------------------------------------

if movimientos_piloto:

    df_movimientos_piloto_v2 = pd.concat(
        movimientos_piloto,
        ignore_index=True
    )

else:

    df_movimientos_piloto_v2 = pd.DataFrame()

# ------------------------------------------------------------
# 6. Resumen de ejecución
# ------------------------------------------------------------

creditos_validos = (
    len(df_piloto_creditos)
    - len(creditos_sin_obligaciones)
    - len(creditos_sin_fecha_valida)
)

print("\nPILOTO V2 GENERADO")

print(
    f"Créditos seleccionados: "
    f"{len(df_piloto_creditos):,}"
)

print(
    f"Créditos simulados: "
    f"{creditos_validos:,}"
)

print(
    f"Sin obligaciones: "
    f"{len(creditos_sin_obligaciones):,}"
)

print(
    f"Sin fecha válida: "
    f"{len(creditos_sin_fecha_valida):,}"
)

print(
    f"Movimientos generados: "
    f"{len(df_movimientos_piloto_v2):,}"
)

# ------------------------------------------------------------
# 7. Distribución de movimientos
# ------------------------------------------------------------

if not df_movimientos_piloto_v2.empty:

    print("\nDistribución de movimientos:")

    display(
        df_movimientos_piloto_v2[
            "resultado_simulado"
        ]
        .value_counts()
        .rename_axis("resultado")
        .reset_index(name="movimientos")
    )

Distribución de perfiles en el piloto:


,perfil,creditos
0,REGULAR,3018
1,RETRASO_OCASIONAL,992
2,IRREGULAR,744
3,DETERIORO_PROGRESIVO,246



Fecha máxima de simulación: 2025-12-31

PILOTO V2 GENERADO
Créditos seleccionados: 5,000
Créditos simulados: 4,925
Sin obligaciones: 75
Sin fecha válida: 0
Movimientos generados: 157,300

Distribución de movimientos:


,resultado,movimientos
0,COMPLETO,119259
1,REGULARIZACION,19559
2,PARCIAL,18482


In [42]:
# ============================================================
# FINPREDICT | EXPLORACIÓN PREVIA A ACTIVIDAD 6
# 06. VALIDACIÓN DEL COMPORTAMIENTO POR PERFIL
# ============================================================

# Asociar cada movimiento con su perfil de simulación
df_movimientos_perfil = (
    df_movimientos_piloto_v2
    .merge(
        df_piloto_creditos[
            ["id_credito", "perfil_simulacion"]
        ],
        on="id_credito",
        how="left"
    )
)

# Contar movimientos por tipo y perfil
tabla_perfiles = (
    df_movimientos_perfil
    .groupby(
        ["perfil_simulacion", "resultado_simulado"]
    )
    .size()
    .unstack(fill_value=0)
)

# Garantizar columnas esperadas
for columna in [
    "COMPLETO",
    "PARCIAL",
    "REGULARIZACION"
]:
    if columna not in tabla_perfiles.columns:
        tabla_perfiles[columna] = 0

# Total de movimientos registrados
tabla_perfiles["TOTAL_MOVIMIENTOS"] = (
    tabla_perfiles[
        ["COMPLETO", "PARCIAL", "REGULARIZACION"]
    ]
    .sum(axis=1)
)

# Porcentajes dentro de cada perfil
tabla_perfiles["% COMPLETO"] = (
    tabla_perfiles["COMPLETO"]
    / tabla_perfiles["TOTAL_MOVIMIENTOS"]
    * 100
).round(2)

tabla_perfiles["% PARCIAL"] = (
    tabla_perfiles["PARCIAL"]
    / tabla_perfiles["TOTAL_MOVIMIENTOS"]
    * 100
).round(2)

tabla_perfiles["% REGULARIZACION"] = (
    tabla_perfiles["REGULARIZACION"]
    / tabla_perfiles["TOTAL_MOVIMIENTOS"]
    * 100
).round(2)

tabla_perfiles = tabla_perfiles[
    [
        "COMPLETO",
        "PARCIAL",
        "REGULARIZACION",
        "TOTAL_MOVIMIENTOS",
        "% COMPLETO",
        "% PARCIAL",
        "% REGULARIZACION"
    ]
]

display(tabla_perfiles)

resultado_simulado,COMPLETO,PARCIAL,REGULARIZACION,TOTAL_MOVIMIENTOS,% COMPLETO,% PARCIAL,% REGULARIZACION
perfil_simulacion,,,,,,,
DETERIORO_PROGRESIVO,4129,1932,463,6524,63.29,29.61,7.10
IRREGULAR,12835,6147,2166,21148,60.69,29.07,10.24
REGULAR,78729,5995,12881,97605,80.66,6.14,13.20
RETRASO_OCASIONAL,23566,4408,4049,32023,73.59,13.77,12.64


In [43]:
# ============================================================
# FINPREDICT | EXPLORACIÓN PREVIA A ACTIVIDAD 6
# 07. LOCALIZAR LÓGICA ORIGINAL DEL TARGET
# ============================================================

import nbformat

ruta_notebook_original = (
    RUTA_REPO /
    "fuentes" /
    "01_construccion_dataset.ipynb"
)

with open(
    ruta_notebook_original,
    "r",
    encoding="utf-8"
) as f:
    nb_original = nbformat.read(
        f,
        as_version=4
    )

terminos_busqueda = [
    "incumplimiento_90d",
    "dias_atraso",
    "mora_90"
]

for i, celda in enumerate(nb_original.cells):

    if celda.cell_type != "code":
        continue

    if any(
        termino in celda.source
        for termino in terminos_busqueda
    ):
        print("\n" + "=" * 90)
        print(f"CELDA ORIGINAL {i}")
        print("=" * 90)
        print(celda.source)


CELDA ORIGINAL 32

# ============================================================
# FINPREDICT | ACTIVIDAD 5
# 17. Cálculo histórico de días de atraso mediante FIFO
# ============================================================

# Obligaciones del contrato de prueba
obligaciones = (
    df_prueba_fechas[
        ["numero_pago", "fecha_vencimiento", "importe_exigible"]
    ]
    .sort_values("fecha_vencimiento")
    .reset_index(drop=True)
)

# Movimientos con pago efectivo
movimientos = (
    df_prueba_fechas.loc[
        df_prueba_fechas["importe_pagado"] > 0,
        ["fecha_pago", "importe_pagado"]
    ]
    .sort_values("fecha_pago")
    .reset_index(drop=True)
)

def calcular_morosidad_fifo(fecha_evaluacion):
    """Calcula obligaciones vencidas y días de atraso
    utilizando exclusivamente pagos hasta la fecha evaluada.
    """

    # Obligaciones vencidas a la fecha evaluada
    vencidas = obligaciones.loc[
        obligaciones["fecha_vencimiento"] <= fecha_evaluacion
    ].co

In [44]:
# ============================================================
# FINPREDICT | EXPLORACIÓN PREVIA A ACTIVIDAD 6
# 07. CONSTRUCCIÓN DEL TARGET V2 EN EL PILOTO
# ============================================================

FECHA_CORTE = pd.Timestamp("2025-09-30")
FECHA_EVALUACION_INICIO = pd.Timestamp("2025-10-01")
FECHA_EVALUACION_FIN = pd.Timestamp("2025-12-31")

TOLERANCIA = 0.01


# ------------------------------------------------------------
# 1. Créditos realmente utilizables en el piloto
# ------------------------------------------------------------

ids_piloto = set(
    df_piloto_creditos["id_credito"]
)

obligaciones_piloto = (
    df_obligaciones.loc[
        df_obligaciones["id_credito"].isin(ids_piloto),
        [
            "id_credito",
            "fecha_vencimiento",
            "importe_exigible"
        ]
    ]
    .dropna(subset=["fecha_vencimiento"])
    .sort_values(
        ["id_credito", "fecha_vencimiento"]
    )
    .copy()
)

movimientos_piloto = (
    df_movimientos_piloto_v2[
        [
            "id_credito",
            "fecha_movimiento",
            "importe"
        ]
    ]
    .sort_values(
        ["id_credito", "fecha_movimiento"]
    )
    .copy()
)


# ------------------------------------------------------------
# 2. Agrupar una sola vez para evitar búsquedas repetitivas
# ------------------------------------------------------------

grupos_obligaciones = {
    credito: grupo.reset_index(drop=True)
    for credito, grupo
    in obligaciones_piloto.groupby("id_credito")
}

grupos_movimientos = {
    credito: grupo.reset_index(drop=True)
    for credito, grupo
    in movimientos_piloto.groupby("id_credito")
}


# ------------------------------------------------------------
# 3. Estado del crédito al corte mediante FIFO
# ------------------------------------------------------------

def calcular_estado_corte_v2(
    obligaciones,
    movimientos
):

    vencidas = obligaciones.loc[
        obligaciones["fecha_vencimiento"] <= FECHA_CORTE
    ].copy()

    if vencidas.empty:
        return 0.0, 0

    pagado = movimientos.loc[
        movimientos["fecha_movimiento"] <= FECHA_CORTE,
        "importe"
    ].sum()

    importes = vencidas[
        "importe_exigible"
    ].to_numpy(dtype=float)

    acumulado = np.cumsum(importes)

    acumulado_anterior = (
        acumulado - importes
    )

    pendiente = np.clip(
        acumulado - pagado,
        0,
        importes
    )

    saldo_vencido = float(
        pendiente.sum()
    )

    indices_pendientes = np.flatnonzero(
        pendiente > TOLERANCIA
    )

    if len(indices_pendientes) == 0:
        dias_atraso = 0

    else:
        primera_fecha = vencidas.iloc[
            indices_pendientes[0]
        ]["fecha_vencimiento"]

        dias_atraso = max(
            (FECHA_CORTE - primera_fecha).days,
            0
        )

    return round(saldo_vencido, 2), dias_atraso


# ------------------------------------------------------------
# 4. Construir población predictiva
# ------------------------------------------------------------

estado_corte = []

for id_credito, obligaciones in grupos_obligaciones.items():

    movimientos = grupos_movimientos.get(
        id_credito,
        pd.DataFrame(
            columns=[
                "id_credito",
                "fecha_movimiento",
                "importe"
            ]
        )
    )

    saldo, dias = calcular_estado_corte_v2(
        obligaciones,
        movimientos
    )

    estado_corte.append({
        "id_credito": id_credito,
        "saldo_vencido_corte": saldo,
        "dias_atraso_corte": dias,
        "mora_90_corte": int(dias >= 90)
    })


df_estado_corte_v2 = pd.DataFrame(
    estado_corte
)

df_poblacion_predictiva_v2 = (
    df_estado_corte_v2.loc[
        df_estado_corte_v2["mora_90_corte"] == 0
    ]
    .copy()
)


# ------------------------------------------------------------
# 5. Evaluar incumplimiento futuro
# ------------------------------------------------------------

def evaluar_target_v2(
    obligaciones,
    movimientos
):

    fechas_vencimiento = obligaciones[
        "fecha_vencimiento"
    ].to_numpy(dtype="datetime64[ns]")

    importes = obligaciones[
        "importe_exigible"
    ].to_numpy(dtype=float)

    acumulado_exigible = np.cumsum(
        importes
    )

    fechas_pago = movimientos[
        "fecha_movimiento"
    ].to_numpy(dtype="datetime64[ns]")

    importes_pago = movimientos[
        "importe"
    ].to_numpy(dtype=float)

    acumulado_pagos = np.cumsum(
        importes_pago
    )

    # Fechas en las que una obligación puede alcanzar 90 días
    fechas_revision = (
        obligaciones["fecha_vencimiento"]
        + pd.Timedelta(days=90)
    )

    fechas_revision = fechas_revision.loc[
        (fechas_revision >= FECHA_EVALUACION_INICIO)
        & (fechas_revision <= FECHA_EVALUACION_FIN)
    ]

    # Revisar también el inicio de la ventana
    fechas_revision = sorted(
        set(
            [FECHA_EVALUACION_INICIO]
            + fechas_revision.tolist()
        )
    )

    max_atraso = 0

    for fecha in fechas_revision:

        fecha_np = np.datetime64(fecha)

        n_obligaciones = np.searchsorted(
            fechas_vencimiento,
            fecha_np,
            side="right"
        )

        if n_obligaciones == 0:
            continue

        n_pagos = np.searchsorted(
            fechas_pago,
            fecha_np,
            side="right"
        )

        pagado = (
            acumulado_pagos[n_pagos - 1]
            if n_pagos > 0
            else 0.0
        )

        idx_pendiente = np.searchsorted(
            acumulado_exigible[:n_obligaciones],
            pagado + TOLERANCIA,
            side="right"
        )

        if idx_pendiente >= n_obligaciones:
            continue

        dias = int(
            (
                fecha_np
                - fechas_vencimiento[idx_pendiente]
            )
            / np.timedelta64(1, "D")
        )

        max_atraso = max(
            max_atraso,
            dias
        )

        if max_atraso >= 90:
            break

    return max_atraso, int(
        max_atraso >= 90
    )


# ------------------------------------------------------------
# 6. Construcción del target
# ------------------------------------------------------------

resultados_target = []

for id_credito in df_poblacion_predictiva_v2[
    "id_credito"
]:

    obligaciones = grupos_obligaciones[
        id_credito
    ]

    movimientos = grupos_movimientos.get(
        id_credito,
        pd.DataFrame(
            columns=[
                "id_credito",
                "fecha_movimiento",
                "importe"
            ]
        )
    )

    max_atraso, incumplimiento = (
        evaluar_target_v2(
            obligaciones,
            movimientos
        )
    )

    resultados_target.append({
        "id_credito": id_credito,
        "max_atraso_futuro": max_atraso,
        "incumplimiento_90d": incumplimiento
    })


df_target_piloto_v2 = pd.DataFrame(
    resultados_target
)


# ------------------------------------------------------------
# 7. Resumen
# ------------------------------------------------------------

print("FINPREDICT | TARGET PILOTO V2")
print("-" * 60)

print(
    f"Créditos con obligaciones: "
    f"{len(df_estado_corte_v2):,}"
)

print(
    f"Mora >= 90 días al corte: "
    f"{df_estado_corte_v2['mora_90_corte'].sum():,}"
)

print(
    f"Población predictiva: "
    f"{len(df_poblacion_predictiva_v2):,}"
)

print(
    f"Nuevos incumplimientos: "
    f"{df_target_piloto_v2['incumplimiento_90d'].sum():,}"
)

print(
    f"Tasa de incumplimiento V2: "
    f"{df_target_piloto_v2['incumplimiento_90d'].mean():.2%}"
)

print("-" * 60)

FINPREDICT | TARGET PILOTO V2
------------------------------------------------------------
Créditos con obligaciones: 4,925
Mora >= 90 días al corte: 487
Población predictiva: 4,438
Nuevos incumplimientos: 355
Tasa de incumplimiento V2: 8.00%
------------------------------------------------------------


In [45]:
# ============================================================
# FINPREDICT | EXPLORACIÓN PREVIA A ACTIVIDAD 6
# 08. RIESGO V2 SEGÚN DÍAS DE ATRASO AL CORTE
# ============================================================

# ------------------------------------------------------------
# 1. Integrar estado al corte con target futuro
# ------------------------------------------------------------

df_validacion_v2 = (
    df_poblacion_predictiva_v2[
        [
            "id_credito",
            "saldo_vencido_corte",
            "dias_atraso_corte"
        ]
    ]
    .merge(
        df_target_piloto_v2[
            [
                "id_credito",
                "max_atraso_futuro",
                "incumplimiento_90d"
            ]
        ],
        on="id_credito",
        how="inner",
        validate="one_to_one"
    )
)

# ------------------------------------------------------------
# 2. Crear rangos comparables con V1
# ------------------------------------------------------------

df_validacion_v2["rango_atraso"] = pd.cut(
    df_validacion_v2["dias_atraso_corte"],
    bins=[
        -1,
        0,
        15,
        30,
        45,
        60,
        75,
        90
    ],
    labels=[
        "0 días",
        "1-15 días",
        "16-30 días",
        "31-45 días",
        "46-60 días",
        "61-75 días",
        "76-90 días"
    ]
)

# ------------------------------------------------------------
# 3. Calcular tasa de incumplimiento futura
# ------------------------------------------------------------

riesgo_por_atraso_v2 = (
    df_validacion_v2
    .groupby(
        "rango_atraso",
        observed=False
    )
    .agg(
        registros=(
            "incumplimiento_90d",
            "size"
        ),
        incumplimientos=(
            "incumplimiento_90d",
            "sum"
        ),
        tasa_incumplimiento=(
            "incumplimiento_90d",
            "mean"
        )
    )
    .reset_index()
)

riesgo_por_atraso_v2[
    "tasa_incumplimiento_%"
] = (
    riesgo_por_atraso_v2[
        "tasa_incumplimiento"
    ] * 100
).round(2)

# ------------------------------------------------------------
# 4. Mostrar resultado
# ------------------------------------------------------------

print("FINPREDICT | RIESGO POR ATRASO — PILOTO V2")
print("-" * 65)

display(
    riesgo_por_atraso_v2[
        [
            "rango_atraso",
            "registros",
            "incumplimientos",
            "tasa_incumplimiento_%"
        ]
    ]
)

print("-" * 65)
print(
    f"Prevalencia general V2: "
    f"{df_validacion_v2['incumplimiento_90d'].mean():.2%}"
)

FINPREDICT | RIESGO POR ATRASO — PILOTO V2
-----------------------------------------------------------------


,rango_atraso,registros,incumplimientos,tasa_incumplimiento_%
0,0 días,1188,1,0.08
1,1-15 días,1139,6,0.53
2,16-30 días,1239,5,0.40
3,31-45 días,352,57,16.19
4,46-60 días,262,74,28.24
5,61-75 días,134,90,67.16
6,76-90 días,124,122,98.39


-----------------------------------------------------------------
Prevalencia general V2: 8.00%


In [46]:
# ============================================================
# FINPREDICT | EXPLORACIÓN PREVIA A ACTIVIDAD 6
# 09. EVALUACIÓN DE POBLACIÓN PREVENTIVA
# ============================================================

# ------------------------------------------------------------
# 1. Separar población por etapa de riesgo
# ------------------------------------------------------------

df_preventivo_v2 = df_validacion_v2.loc[
    df_validacion_v2["dias_atraso_corte"] <= 60
].copy()

df_cobranza_intensiva_v2 = df_validacion_v2.loc[
    df_validacion_v2["dias_atraso_corte"].between(
        61,
        89,
        inclusive="both"
    )
].copy()


# ------------------------------------------------------------
# 2. Resumen de poblaciones
# ------------------------------------------------------------

resumen_segmentacion = pd.DataFrame({
    "segmento": [
        "Modelo preventivo (0-60 días)",
        "Cobranza intensiva (61-89 días)"
    ],
    "registros": [
        len(df_preventivo_v2),
        len(df_cobranza_intensiva_v2)
    ],
    "incumplimientos": [
        int(
            df_preventivo_v2[
                "incumplimiento_90d"
            ].sum()
        ),
        int(
            df_cobranza_intensiva_v2[
                "incumplimiento_90d"
            ].sum()
        )
    ],
    "tasa_incumplimiento_%": [
        round(
            df_preventivo_v2[
                "incumplimiento_90d"
            ].mean() * 100,
            2
        ),
        round(
            df_cobranza_intensiva_v2[
                "incumplimiento_90d"
            ].mean() * 100,
            2
        )
    ]
})

print("FINPREDICT | SEGMENTACIÓN OPERATIVA V2")
print("-" * 65)

display(resumen_segmentacion)

print(
    f"\nCobertura del modelo preventivo: "
    f"{len(df_preventivo_v2) / len(df_validacion_v2):.2%}"
)

FINPREDICT | SEGMENTACIÓN OPERATIVA V2
-----------------------------------------------------------------


,segmento,registros,incumplimientos,tasa_incumplimiento_%
0,Modelo preventivo (0-60 días),4180,143,3.42
1,Cobranza intensiva (61-89 días),258,212,82.17



Cobertura del modelo preventivo: 94.19%


In [47]:
# ============================================================
# FINPREDICT | EXPLORACIÓN PREVIA A ACTIVIDAD 6
# 10. COMPARACIÓN V1 VS V2 EN POBLACIÓN PREVENTIVA
# ============================================================

# ------------------------------------------------------------
# 1. Población preventiva V1
#    Mismo criterio utilizado en V2: máximo 60 días de atraso
# ------------------------------------------------------------

df_preventivo_v1 = df_desarrollo_v1.loc[
    df_desarrollo_v1["dias_atraso"] <= 60
].copy()

# ------------------------------------------------------------
# 2. Resumen general comparable
# ------------------------------------------------------------

comparacion_general = pd.DataFrame({
    "version": [
        "V1",
        "V2 piloto"
    ],
    "registros": [
        len(df_preventivo_v1),
        len(df_preventivo_v2)
    ],
    "incumplimientos": [
        int(
            df_preventivo_v1[
                "incumplimiento_90d"
            ].sum()
        ),
        int(
            df_preventivo_v2[
                "incumplimiento_90d"
            ].sum()
        )
    ],
    "tasa_incumplimiento_%": [
        round(
            df_preventivo_v1[
                "incumplimiento_90d"
            ].mean() * 100,
            2
        ),
        round(
            df_preventivo_v2[
                "incumplimiento_90d"
            ].mean() * 100,
            2
        )
    ]
})

print("FINPREDICT | COMPARACIÓN PREVENTIVA V1 VS V2")
print("-" * 65)

display(comparacion_general)


# ------------------------------------------------------------
# 3. Tasas V1 por rango
# ------------------------------------------------------------

df_preventivo_v1["rango_atraso"] = pd.cut(
    df_preventivo_v1["dias_atraso"],
    bins=[-1, 0, 15, 30, 45, 60],
    labels=[
        "0 días",
        "1-15 días",
        "16-30 días",
        "31-45 días",
        "46-60 días"
    ]
)

riesgo_v1 = (
    df_preventivo_v1
    .groupby(
        "rango_atraso",
        observed=False
    )
    .agg(
        registros=("incumplimiento_90d", "size"),
        incumplimientos=("incumplimiento_90d", "sum"),
        tasa=("incumplimiento_90d", "mean")
    )
    .reset_index()
)

riesgo_v1["tasa_V1_%"] = (
    riesgo_v1["tasa"] * 100
).round(2)


# ------------------------------------------------------------
# 4. Tasas V2 por rango
# ------------------------------------------------------------

riesgo_v2 = (
    df_preventivo_v2
    .assign(
        rango_atraso=pd.cut(
            df_preventivo_v2["dias_atraso_corte"],
            bins=[-1, 0, 15, 30, 45, 60],
            labels=[
                "0 días",
                "1-15 días",
                "16-30 días",
                "31-45 días",
                "46-60 días"
            ]
        )
    )
    .groupby(
        "rango_atraso",
        observed=False
    )
    .agg(
        registros=("incumplimiento_90d", "size"),
        incumplimientos=("incumplimiento_90d", "sum"),
        tasa=("incumplimiento_90d", "mean")
    )
    .reset_index()
)

riesgo_v2["tasa_V2_%"] = (
    riesgo_v2["tasa"] * 100
).round(2)


# ------------------------------------------------------------
# 5. Comparación directa
# ------------------------------------------------------------

comparacion_rangos = (
    riesgo_v1[
        [
            "rango_atraso",
            "tasa_V1_%"
        ]
    ]
    .merge(
        riesgo_v2[
            [
                "rango_atraso",
                "tasa_V2_%"
            ]
        ],
        on="rango_atraso",
        how="outer"
    )
)

comparacion_rangos["diferencia_pp"] = (
    comparacion_rangos["tasa_V2_%"]
    - comparacion_rangos["tasa_V1_%"]
).round(2)

print("\nRIESGO FUTURO POR DÍAS DE ATRASO")
display(comparacion_rangos)

FINPREDICT | COMPARACIÓN PREVENTIVA V1 VS V2
-----------------------------------------------------------------


,version,registros,incumplimientos,tasa_incumplimiento_%
0,V1,337604,9476,2.81
1,V2 piloto,4180,143,3.42



RIESGO FUTURO POR DÍAS DE ATRASO


,rango_atraso,tasa_V1_%,tasa_V2_%,diferencia_pp
0,0 días,0.02,0.08,0.06
1,1-15 días,0.67,0.53,-0.14
2,16-30 días,0.98,0.40,-0.58
3,31-45 días,27.23,16.19,-11.04
4,46-60 días,39.79,28.24,-11.55


In [48]:
# ============================================================
# FINPREDICT | EXPLORACIÓN PREVIA A ACTIVIDAD 6
# 11. GENERACIÓN COMPLETA OPTIMIZADA DE MOVIMIENTOS V2
# ============================================================

# ------------------------------------------------------------
# 1. Asignar perfiles a los 50,000 contratos
# ------------------------------------------------------------

df_perfiles_v2 = asignar_perfiles_v2(
    df_contratos,
    seed=SEED
)

# ------------------------------------------------------------
# 2. Preparar obligaciones una sola vez
# ------------------------------------------------------------

df_obligaciones["fecha_vencimiento"] = pd.to_datetime(
    df_obligaciones["fecha_vencimiento"],
    errors="coerce"
)

FECHA_FIN_V2 = df_obligaciones[
    "fecha_vencimiento"
].max()

# Eliminar únicamente obligaciones sin fecha válida
obligaciones_validas = (
    df_obligaciones
    .dropna(subset=["fecha_vencimiento"])
    .sort_values(
        ["id_credito", "numero_pago"]
    )
    .copy()
)

# Crear diccionario una sola vez
grupos_obligaciones_v2 = {
    credito: grupo.reset_index(drop=True)
    for credito, grupo
    in obligaciones_validas.groupby(
        "id_credito",
        sort=False
    )
}

print(
    f"Contratos con obligaciones válidas: "
    f"{len(grupos_obligaciones_v2):,}"
)

print(
    f"Fecha máxima de simulación: "
    f"{FECHA_FIN_V2.date()}"
)

# ------------------------------------------------------------
# 3. Simular cartera completa
# ------------------------------------------------------------

movimientos_v2 = []
sin_obligaciones_v2 = []

for fila in df_perfiles_v2.itertuples(index=False):

    id_credito = fila.id_credito
    perfil = fila.perfil_simulacion

    obligaciones_credito = (
        grupos_obligaciones_v2.get(id_credito)
    )

    if obligaciones_credito is None:
        sin_obligaciones_v2.append(id_credito)
        continue

    numero_credito = int(
        id_credito.replace("HIP", "")
    )

    rng_credito = np.random.default_rng(
        SEED + numero_credito
    )

    movimientos_credito = (
        simular_movimientos_finpredict_v2(
            obligaciones_credito,
            perfil,
            rng_credito,
            FECHA_FIN_V2
        )
    )

    if not movimientos_credito.empty:
        movimientos_v2.append(
            movimientos_credito
        )

# ------------------------------------------------------------
# 4. Consolidar movimientos
# ------------------------------------------------------------

df_movimientos_v2 = (
    pd.concat(
        movimientos_v2,
        ignore_index=True
    )
    .sort_values(
        [
            "id_credito",
            "fecha_movimiento",
            "numero_pago_origen"
        ],
        na_position="last",
        kind="stable"
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 5. Validaciones
# ------------------------------------------------------------

assert not df_movimientos_v2.empty

assert df_movimientos_v2[
    "fecha_movimiento"
].notna().all()

assert (
    df_movimientos_v2["importe"] > 0
).all()

assert (
    df_movimientos_v2["fecha_movimiento"]
    <= FECHA_FIN_V2
).all()

# ------------------------------------------------------------
# 6. Resumen
# ------------------------------------------------------------

print("\nFINPREDICT | MOVIMIENTOS V2 COMPLETOS")
print("-" * 65)

print(
    f"Contratos originales: "
    f"{len(df_contratos):,}"
)

print(
    f"Contratos sin obligaciones: "
    f"{len(sin_obligaciones_v2):,}"
)

print(
    f"Contratos simulables: "
    f"{len(grupos_obligaciones_v2):,}"
)

print(
    f"Movimientos generados: "
    f"{len(df_movimientos_v2):,}"
)

print("\nDistribución de movimientos:")

display(
    df_movimientos_v2[
        "resultado_simulado"
    ]
    .value_counts()
    .rename_axis("resultado")
    .reset_index(name="movimientos")
)

Contratos con obligaciones válidas: 49,093
Fecha máxima de simulación: 2025-12-31


/tmp/ipykernel_2736/1833807246.py:343: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  historial = pd.concat(



FINPREDICT | MOVIMIENTOS V2 COMPLETOS
-----------------------------------------------------------------
Contratos originales: 50,000
Contratos sin obligaciones: 907
Contratos simulables: 49,093
Movimientos generados: 1,578,515

Distribución de movimientos:


,resultado,movimientos
0,COMPLETO,1197798
1,REGULARIZACION,197001
2,PARCIAL,183716


In [49]:
# ============================================================
# FINPREDICT | EXPLORACIÓN PREVIA A ACTIVIDAD 6
# 12. VALIDACIÓN COMPLETA DEL TARGET V2
# ============================================================

FECHA_CORTE = pd.Timestamp("2025-09-30")
FECHA_EVALUACION_INICIO = pd.Timestamp("2025-10-01")
FECHA_EVALUACION_FIN = pd.Timestamp("2025-12-31")

TOLERANCIA = 0.01


# ------------------------------------------------------------
# 1. Universo válido V2
# ------------------------------------------------------------

ids_validos_v2 = set(
    obligaciones_validas["id_credito"].unique()
)

obligaciones_v2 = (
    obligaciones_validas.loc[
        obligaciones_validas["id_credito"].isin(ids_validos_v2),
        [
            "id_credito",
            "fecha_vencimiento",
            "importe_exigible"
        ]
    ]
    .sort_values(
        ["id_credito", "fecha_vencimiento"]
    )
    .copy()
)

movimientos_v2_base = (
    df_movimientos_v2[
        [
            "id_credito",
            "fecha_movimiento",
            "importe"
        ]
    ]
    .sort_values(
        ["id_credito", "fecha_movimiento"]
    )
    .copy()
)


# ------------------------------------------------------------
# 2. Estado de morosidad al 30/09/2025
# ------------------------------------------------------------

obligaciones_corte = obligaciones_v2.loc[
    obligaciones_v2["fecha_vencimiento"] <= FECHA_CORTE
].copy()

obligaciones_corte["exigible_acumulado"] = (
    obligaciones_corte
    .groupby("id_credito")["importe_exigible"]
    .cumsum()
)

pagos_corte = (
    movimientos_v2_base.loc[
        movimientos_v2_base["fecha_movimiento"] <= FECHA_CORTE
    ]
    .groupby("id_credito")["importe"]
    .sum()
    .rename("total_pagado")
)

obligaciones_corte = obligaciones_corte.join(
    pagos_corte,
    on="id_credito"
)

obligaciones_corte["total_pagado"] = (
    obligaciones_corte["total_pagado"]
    .fillna(0)
)

obligaciones_corte["saldo_pendiente"] = np.clip(
    obligaciones_corte["exigible_acumulado"]
    - obligaciones_corte["total_pagado"],
    0,
    obligaciones_corte["importe_exigible"]
).round(2)

pendientes = obligaciones_corte.loc[
    obligaciones_corte["saldo_pendiente"] > TOLERANCIA
]

resumen_morosidad = (
    pendientes
    .groupby("id_credito")
    .agg(
        saldo_vencido_corte=(
            "saldo_pendiente",
            "sum"
        ),
        vencimiento_antiguo=(
            "fecha_vencimiento",
            "min"
        )
    )
    .reset_index()
)

df_estado_v2_completo = (
    pd.DataFrame({
        "id_credito": sorted(ids_validos_v2)
    })
    .merge(
        resumen_morosidad,
        on="id_credito",
        how="left",
        validate="one_to_one"
    )
)

df_estado_v2_completo["saldo_vencido_corte"] = (
    df_estado_v2_completo[
        "saldo_vencido_corte"
    ]
    .fillna(0)
    .round(2)
)

df_estado_v2_completo["dias_atraso_corte"] = (
    FECHA_CORTE
    - df_estado_v2_completo["vencimiento_antiguo"]
).dt.days.fillna(0).astype(int)

df_estado_v2_completo["mora_90_corte"] = (
    df_estado_v2_completo["dias_atraso_corte"] >= 90
).astype(int)


# ------------------------------------------------------------
# 3. Población predictiva
# ------------------------------------------------------------

df_poblacion_v2_completa = (
    df_estado_v2_completo.loc[
        df_estado_v2_completo["mora_90_corte"] == 0
    ]
    .copy()
)

ids_predictivos = set(
    df_poblacion_v2_completa["id_credito"]
)


# ------------------------------------------------------------
# 4. Preparar obligaciones acumuladas
# ------------------------------------------------------------

obligaciones_target = (
    obligaciones_v2.loc[
        obligaciones_v2["id_credito"].isin(ids_predictivos)
    ]
    .copy()
)

obligaciones_target["exigible_acumulado"] = (
    obligaciones_target
    .groupby("id_credito")["importe_exigible"]
    .cumsum()
)

# Solo obligaciones capaces de alcanzar 90 días
# durante la ventana de evaluación
obligaciones_evaluar = obligaciones_target.loc[
    obligaciones_target["fecha_vencimiento"]
    <= FECHA_EVALUACION_FIN - pd.Timedelta(days=90)
].copy()

obligaciones_evaluar["fecha_revision"] = (
    obligaciones_evaluar["fecha_vencimiento"]
    + pd.Timedelta(days=90)
)

obligaciones_evaluar["fecha_revision"] = (
    obligaciones_evaluar["fecha_revision"]
    .clip(lower=FECHA_EVALUACION_INICIO)
)

obligaciones_evaluar = obligaciones_evaluar.loc[
    obligaciones_evaluar["fecha_revision"]
    <= FECHA_EVALUACION_FIN
].copy()


# ------------------------------------------------------------
# 5. Preparar pagos acumulados
# ------------------------------------------------------------

pagos_target = (
    movimientos_v2_base.loc[
        movimientos_v2_base["id_credito"].isin(
            ids_predictivos
        )
    ]
    .copy()
)

pagos_target["pagado_acumulado"] = (
    pagos_target
    .groupby("id_credito")["importe"]
    .cumsum()
)


# ------------------------------------------------------------
# 6. Conciliar pagos disponibles a cada fecha de revisión
# ------------------------------------------------------------

obligaciones_evaluar = (
    obligaciones_evaluar
    .sort_values(
        ["fecha_revision", "id_credito"]
    )
)

pagos_target = (
    pagos_target
    .sort_values(
        ["fecha_movimiento", "id_credito"]
    )
)

conciliacion = pd.merge_asof(
    obligaciones_evaluar,
    pagos_target[
        [
            "id_credito",
            "fecha_movimiento",
            "pagado_acumulado"
        ]
    ],
    left_on="fecha_revision",
    right_on="fecha_movimiento",
    by="id_credito",
    direction="backward",
    allow_exact_matches=True
)

conciliacion["pagado_acumulado"] = (
    conciliacion["pagado_acumulado"]
    .fillna(0)
)

conciliacion["saldo_90_pendiente"] = (
    conciliacion["exigible_acumulado"]
    - conciliacion["pagado_acumulado"]
).clip(lower=0)

conciliacion["evento_90"] = (
    conciliacion["saldo_90_pendiente"]
    > TOLERANCIA
).astype(int)


# ------------------------------------------------------------
# 7. Target final por crédito
# ------------------------------------------------------------

etiquetas_v2 = (
    conciliacion
    .groupby("id_credito")["evento_90"]
    .max()
    .rename("incumplimiento_90d")
    .reset_index()
)

df_target_v2_completo = (
    df_poblacion_v2_completa
    .merge(
        etiquetas_v2,
        on="id_credito",
        how="left",
        validate="one_to_one"
    )
)

df_target_v2_completo["incumplimiento_90d"] = (
    df_target_v2_completo[
        "incumplimiento_90d"
    ]
    .fillna(0)
    .astype(int)
)


# ------------------------------------------------------------
# 8. Segmentación operativa
# ------------------------------------------------------------

df_target_v2_completo["segmento_operativo"] = np.select(
    [
        df_target_v2_completo[
            "dias_atraso_corte"
        ] <= 60,

        df_target_v2_completo[
            "dias_atraso_corte"
        ].between(61, 89)
    ],
    [
        "MODELO_PREVENTIVO",
        "COBRANZA_INTENSIVA"
    ],
    default="REVISAR"
)

resumen_segmentos_v2 = (
    df_target_v2_completo
    .groupby("segmento_operativo")
    .agg(
        registros=(
            "id_credito",
            "count"
        ),
        incumplimientos=(
            "incumplimiento_90d",
            "sum"
        ),
        tasa_incumplimiento=(
            "incumplimiento_90d",
            "mean"
        )
    )
    .reset_index()
)

resumen_segmentos_v2[
    "tasa_incumplimiento_%"
] = (
    resumen_segmentos_v2[
        "tasa_incumplimiento"
    ] * 100
).round(2)


# ------------------------------------------------------------
# 9. Riesgo por rango de atraso
# ------------------------------------------------------------

df_target_v2_completo["rango_atraso"] = pd.cut(
    df_target_v2_completo["dias_atraso_corte"],
    bins=[
        -1,
        0,
        15,
        30,
        45,
        60,
        75,
        90
    ],
    labels=[
        "0 días",
        "1-15 días",
        "16-30 días",
        "31-45 días",
        "46-60 días",
        "61-75 días",
        "76-90 días"
    ]
)

riesgo_completo_v2 = (
    df_target_v2_completo
    .groupby(
        "rango_atraso",
        observed=False
    )
    .agg(
        registros=(
            "incumplimiento_90d",
            "size"
        ),
        incumplimientos=(
            "incumplimiento_90d",
            "sum"
        ),
        tasa=(
            "incumplimiento_90d",
            "mean"
        )
    )
    .reset_index()
)

riesgo_completo_v2[
    "tasa_incumplimiento_%"
] = (
    riesgo_completo_v2["tasa"]
    * 100
).round(2)


# ------------------------------------------------------------
# 10. Resultados
# ------------------------------------------------------------

print("FINPREDICT | VALIDACIÓN COMPLETA V2")
print("-" * 65)

print(
    f"Contratos válidos: "
    f"{len(df_estado_v2_completo):,}"
)

print(
    f"Mora >=90 al corte: "
    f"{df_estado_v2_completo['mora_90_corte'].sum():,}"
)

print(
    f"Población predictiva: "
    f"{len(df_target_v2_completo):,}"
)

print(
    f"Nuevos incumplimientos: "
    f"{df_target_v2_completo['incumplimiento_90d'].sum():,}"
)

print(
    f"Tasa de incumplimiento: "
    f"{df_target_v2_completo['incumplimiento_90d'].mean():.2%}"
)

print("\nSEGMENTACIÓN OPERATIVA")
display(
    resumen_segmentos_v2[
        [
            "segmento_operativo",
            "registros",
            "incumplimientos",
            "tasa_incumplimiento_%"
        ]
    ]
)

print("\nRIESGO POR DÍAS DE ATRASO")
display(
    riesgo_completo_v2[
        [
            "rango_atraso",
            "registros",
            "incumplimientos",
            "tasa_incumplimiento_%"
        ]
    ]
)

FINPREDICT | VALIDACIÓN COMPLETA V2
-----------------------------------------------------------------
Contratos válidos: 49,093
Mora >=90 al corte: 4,842
Población predictiva: 44,251
Nuevos incumplimientos: 3,495
Tasa de incumplimiento: 7.90%

SEGMENTACIÓN OPERATIVA


,segmento_operativo,registros,incumplimientos,tasa_incumplimiento_%
0,COBRANZA_INTENSIVA,2569,2067,80.46
1,MODELO_PREVENTIVO,41682,1428,3.43



RIESGO POR DÍAS DE ATRASO


,rango_atraso,registros,incumplimientos,tasa_incumplimiento_%
0,0 días,11135,5,0.04
1,1-15 días,11671,78,0.67
2,16-30 días,12520,129,1.03
3,31-45 días,3416,489,14.31
4,46-60 días,2940,727,24.73
5,61-75 días,1359,895,65.86
6,76-90 días,1210,1172,96.86


In [50]:
# ============================================================
# FINPREDICT | EXPLORACIÓN PREVIA A ACTIVIDAD 6
# 13. CHECKPOINT DE LA SIMULACIÓN V2
# ============================================================

import json

# ------------------------------------------------------------
# 1. Crear directorio exclusivo para V2
# ------------------------------------------------------------

RUTA_V2 = (
    RUTA_REPO
    / "datos"
    / "exploracion_actividad6"
)

RUTA_V2.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# 2. Guardar movimientos completos
# ------------------------------------------------------------

df_movimientos_v2.to_parquet(
    RUTA_V2 / "movimientos_v2.parquet",
    index=False,
    compression="snappy"
)

# ------------------------------------------------------------
# 3. Guardar perfiles asignados
# ------------------------------------------------------------

df_perfiles_v2.to_parquet(
    RUTA_V2 / "perfiles_v2.parquet",
    index=False,
    compression="snappy"
)

# ------------------------------------------------------------
# 4. Guardar validación final
# ------------------------------------------------------------

df_target_v2_completo.to_parquet(
    RUTA_V2 / "validacion_target_v2.parquet",
    index=False,
    compression="snappy"
)

# ------------------------------------------------------------
# 5. Guardar manifiesto metodológico
# ------------------------------------------------------------

manifiesto_v2 = {

    "version": "V2",

    "objetivo": (
        "Reducir el comportamiento excesivamente determinista "
        "observado en V1 preservando la lógica temporal y "
        "el desbalance natural del riesgo hipotecario."
    ),

    "seed": SEED,

    "contratos_originales": int(
        len(df_contratos)
    ),

    "contratos_con_obligaciones": int(
        len(df_estado_v2_completo)
    ),

    "contratos_sin_obligaciones": int(
        len(sin_obligaciones_v2)
    ),

    "movimientos_generados": int(
        len(df_movimientos_v2)
    ),

    "poblacion_predictiva": int(
        len(df_target_v2_completo)
    ),

    "incumplimientos": int(
        df_target_v2_completo[
            "incumplimiento_90d"
        ].sum()
    ),

    "tasa_incumplimiento": float(
        df_target_v2_completo[
            "incumplimiento_90d"
        ].mean()
    ),

    "regla_modelo_preventivo": (
        "0 a 60 días de atraso al corte"
    ),

    "regla_cobranza_intensiva": (
        "61 a 89 días de atraso al corte"
    ),

    "fecha_corte_validacion": (
        "2025-09-30"
    ),

    "ventana_prediccion": (
        "2025-10-01 a 2025-12-31"
    )
}

with open(
    RUTA_V2 / "manifiesto_v2.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifiesto_v2,
        f,
        ensure_ascii=False,
        indent=4
    )

# ------------------------------------------------------------
# 6. Verificación
# ------------------------------------------------------------

archivos_v2 = sorted(
    archivo.name
    for archivo in RUTA_V2.iterdir()
)

print("FINPREDICT | CHECKPOINT V2")
print("-" * 60)

print(f"Ruta: {RUTA_V2}")

print("\nArchivos guardados:")

for archivo in archivos_v2:
    print(f"✓ {archivo}")

print("-" * 60)
print("Checkpoint V2 guardado correctamente.")

FINPREDICT | CHECKPOINT V2
------------------------------------------------------------
Ruta: /content/FinPredict-Actividad5/datos/exploracion_actividad6

Archivos guardados:
✓ manifiesto_v2.json
✓ movimientos_v2.parquet
✓ perfiles_v2.parquet
✓ validacion_target_v2.parquet
------------------------------------------------------------
Checkpoint V2 guardado correctamente.


In [51]:
# ============================================================
# FINPREDICT | EXPLORACIÓN PREVIA A ACTIVIDAD 6
# 14. RECUPERAR ESTRUCTURA HISTÓRICA DE VENTANAS
# ============================================================

# Usamos la estructura temporal ya validada en V1
columnas_ventana = [
    "id_credito",
    "id_ventana",
    "fecha_corte",
    "inicio_prediccion",
    "fin_prediccion",
    "uso"
]

df_ventanas_v1 = (
    df_analitico_v0[
        columnas_ventana
    ]
    .drop_duplicates()
    .sort_values(
        ["fecha_corte", "id_credito"]
    )
    .reset_index(drop=True)
)

# Resumen de ventanas
resumen_ventanas = (
    df_ventanas_v1
    .groupby(
        [
            "id_ventana",
            "fecha_corte",
            "inicio_prediccion",
            "fin_prediccion",
            "uso"
        ]
    )
    .agg(
        observaciones=("id_credito", "count"),
        creditos=("id_credito", "nunique")
    )
    .reset_index()
)

print("FINPREDICT | ESTRUCTURA HISTÓRICA DE VENTANAS")
print("-" * 70)

display(resumen_ventanas)

print(
    f"\nObservaciones contrato-periodo: "
    f"{len(df_ventanas_v1):,}"
)

print(
    f"Créditos distintos: "
    f"{df_ventanas_v1['id_credito'].nunique():,}"
)

print(
    f"Ventanas temporales: "
    f"{df_ventanas_v1['id_ventana'].nunique()}"
)

FINPREDICT | ESTRUCTURA HISTÓRICA DE VENTANAS
----------------------------------------------------------------------


,id_ventana,fecha_corte,inicio_prediccion,fin_prediccion,uso,observaciones,creditos
0,2,2021-06-30,2021-07-01,2021-09-30,HISTORICO,2505,2505
1,3,2021-09-30,2021-10-01,2021-12-31,HISTORICO,4874,4874
2,4,2021-12-31,2022-01-01,2022-03-31,HISTORICO,7327,7327
3,5,2022-03-31,2022-04-01,2022-06-30,HISTORICO,9649,9649
4,6,2022-06-30,2022-07-01,2022-09-30,HISTORICO,11914,11914
5,7,2022-09-30,2022-10-01,2022-12-31,HISTORICO,14155,14155
6,8,2022-12-31,2023-01-01,2023-03-31,HISTORICO,16409,16409
7,9,2023-03-31,2023-04-01,2023-06-30,HISTORICO,18608,18608
8,10,2023-06-30,2023-07-01,2023-09-30,HISTORICO,20785,20785
9,11,2023-09-30,2023-10-01,2023-12-31,HISTORICO,22812,22812



Observaciones contrato-periodo: 390,661
Créditos distintos: 44,913
Ventanas temporales: 18


In [52]:
# ============================================================
# FINPREDICT | EXPLORACIÓN PREVIA A ACTIVIDAD 6
# 15. AUDITORÍA DE LA REGLA TEMPORAL DE ELEGIBILIDAD V1
# ============================================================

# Fechas únicas de las ventanas ya validadas
df_ventanas_base = (
    resumen_ventanas[
        [
            "id_ventana",
            "fecha_corte",
            "inicio_prediccion",
            "fin_prediccion",
            "uso"
        ]
    ]
    .copy()
)

# Asegurar fechas
df_contratos["fecha_apertura"] = pd.to_datetime(
    df_contratos["fecha_apertura"],
    errors="coerce"
)

resultados_regla = []

for ventana in df_ventanas_base.itertuples(index=False):

    # Conteo real observado en V1
    real_v1 = int(
        resumen_ventanas.loc[
            resumen_ventanas["id_ventana"] == ventana.id_ventana,
            "creditos"
        ].iloc[0]
    )

    # Hipótesis A:
    # crédito abierto a más tardar en la fecha de corte
    candidatos_corte = int(
        (
            df_contratos["fecha_apertura"]
            <= ventana.fecha_corte
        ).sum()
    )

    # Hipótesis B:
    # al menos 3 meses de antigüedad al corte
    candidatos_3m = int(
        (
            df_contratos["fecha_apertura"]
            <= ventana.fecha_corte - pd.DateOffset(months=3)
        ).sum()
    )

    # Hipótesis C:
    # al menos 6 meses de antigüedad al corte
    candidatos_6m = int(
        (
            df_contratos["fecha_apertura"]
            <= ventana.fecha_corte - pd.DateOffset(months=6)
        ).sum()
    )

    resultados_regla.append({
        "id_ventana": ventana.id_ventana,
        "fecha_corte": ventana.fecha_corte,
        "real_V1": real_v1,
        "abiertos_al_corte": candidatos_corte,
        "antiguedad_3m": candidatos_3m,
        "antiguedad_6m": candidatos_6m,
        "dif_corte": candidatos_corte - real_v1,
        "dif_3m": candidatos_3m - real_v1,
        "dif_6m": candidatos_6m - real_v1
    })

df_auditoria_regla = pd.DataFrame(resultados_regla)

display(df_auditoria_regla)

,id_ventana,fecha_corte,real_V1,abiertos_al_corte,antiguedad_3m,antiguedad_6m,dif_corte,dif_3m,dif_6m
0,2,2021-06-30,2505,4944,2484,0,2439,-21,-2505
1,3,2021-09-30,4874,7499,4944,2484,2625,70,-2390
2,4,2021-12-31,7327,10016,7499,4944,2689,172,-2383
3,5,2022-03-31,9649,12529,10016,7499,2880,367,-2150
4,6,2022-06-30,11914,15016,12499,9981,3102,585,-1933
5,7,2022-09-30,14155,17575,15016,12499,3420,861,-1656
6,8,2022-12-31,16409,19975,17575,15016,3566,1166,-1393
7,9,2023-03-31,18608,22476,19975,17575,3868,1367,-1033
8,10,2023-06-30,20785,24923,22445,19955,4138,1660,-830
9,11,2023-09-30,22812,27506,24923,22445,4694,2111,-367


In [53]:
# ============================================================
# FINPREDICT | EXPLORACIÓN PREVIA A ACTIVIDAD 6
# 16. IDENTIFICAR REGLA EXACTA DE ELEGIBILIDAD HISTÓRICA
# ============================================================

# ------------------------------------------------------------
# 1. Último vencimiento disponible por contrato
# ------------------------------------------------------------

df_ultimo_vencimiento = (
    df_obligaciones
    .groupby("id_credito", as_index=False)
    .agg(
        ultimo_vencimiento=("fecha_vencimiento", "max")
    )
)

df_contratos_audit = (
    df_contratos[
        ["id_credito", "fecha_apertura"]
    ]
    .merge(
        df_ultimo_vencimiento,
        on="id_credito",
        how="left",
        validate="one_to_one"
    )
)

# ------------------------------------------------------------
# 2. Probar reglas combinadas
# ------------------------------------------------------------

resultados = []

for ventana in df_ventanas_base.itertuples(index=False):

    real_v1 = int(
        resumen_ventanas.loc[
            resumen_ventanas["id_ventana"] == ventana.id_ventana,
            "creditos"
        ].iloc[0]
    )

    antiguedad_3m = (
        df_contratos_audit["fecha_apertura"]
        <= ventana.fecha_corte - pd.DateOffset(months=3)
    )

    # A. Antigüedad + crédito activo al corte
    regla_a = (
        antiguedad_3m
        & (
            df_contratos_audit["ultimo_vencimiento"]
            >= ventana.fecha_corte
        )
    )

    # B. Antigüedad + crédito activo durante el trimestre futuro
    regla_b = (
        antiguedad_3m
        & (
            df_contratos_audit["ultimo_vencimiento"]
            >= ventana.fin_prediccion
        )
    )

    # C. Antigüedad + al menos 90 días de vida futura
    regla_c = (
        antiguedad_3m
        & (
            df_contratos_audit["ultimo_vencimiento"]
            >= ventana.fecha_corte + pd.Timedelta(days=90)
        )
    )

    resultados.append({
        "id_ventana": ventana.id_ventana,
        "fecha_corte": ventana.fecha_corte,
        "real_V1": real_v1,

        "3m_activo_corte": int(regla_a.sum()),
        "dif_A": int(regla_a.sum()) - real_v1,

        "3m_activo_fin_pred": int(regla_b.sum()),
        "dif_B": int(regla_b.sum()) - real_v1,

        "3m_vida_90d": int(regla_c.sum()),
        "dif_C": int(regla_c.sum()) - real_v1
    })

df_auditoria_elegibilidad = pd.DataFrame(resultados)

display(df_auditoria_elegibilidad)

,id_ventana,fecha_corte,real_V1,3m_activo_corte,dif_A,3m_activo_fin_pred,dif_B,3m_vida_90d,dif_C
0,2,2021-06-30,2505,2484,-21,2484,-21,2484,-21
1,3,2021-09-30,4874,4944,70,4944,70,4944,70
2,4,2021-12-31,7327,7499,172,7499,172,7499,172
3,5,2022-03-31,9649,10016,367,10016,367,10016,367
4,6,2022-06-30,11914,12499,585,12499,585,12499,585
5,7,2022-09-30,14155,15016,861,15016,861,15016,861
6,8,2022-12-31,16409,17575,1166,17575,1166,17575,1166
7,9,2023-03-31,18608,19975,1367,19975,1367,19975,1367
8,10,2023-06-30,20785,22445,1660,22445,1660,22445,1660
9,11,2023-09-30,22812,24923,2111,24923,2111,24923,2111


In [54]:
# ============================================================
# FINPREDICT | EXPLORACIÓN PREVIA A ACTIVIDAD 6
# 17. FIJAR COHORTE ANALÍTICA V2
# ============================================================

# La V2 conserva exactamente la misma estructura
# contrato-periodo utilizada y validada en V1.

columnas_cohorte = [
    "id_credito",
    "id_ventana",
    "fecha_corte",
    "inicio_prediccion",
    "fin_prediccion",
    "uso"
]

df_contrato_periodo_v2 = (
    df_analitico_v0[
        columnas_cohorte
    ]
    .copy()
    .sort_values(
        ["fecha_corte", "id_credito"]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Validaciones de integridad
# ------------------------------------------------------------

assert len(df_contrato_periodo_v2) == 390_661

assert not df_contrato_periodo_v2.duplicated(
    ["id_credito", "fecha_corte"]
).any()

assert (
    df_contrato_periodo_v2["fecha_corte"]
    < df_contrato_periodo_v2["inicio_prediccion"]
).all()

assert (
    df_contrato_periodo_v2["inicio_prediccion"]
    <= df_contrato_periodo_v2["fin_prediccion"]
).all()

# Comprobar cobertura dentro de la simulación V2
ids_cohorte = set(
    df_contrato_periodo_v2["id_credito"]
)

ids_simulables_v2 = set(
    obligaciones_validas["id_credito"]
)

ids_sin_cobertura_v2 = (
    ids_cohorte - ids_simulables_v2
)

print("FINPREDICT | COHORTE ANALÍTICA V2")
print("-" * 65)

print(
    f"Observaciones contrato-periodo: "
    f"{len(df_contrato_periodo_v2):,}"
)

print(
    f"Créditos distintos: "
    f"{df_contrato_periodo_v2['id_credito'].nunique():,}"
)

print(
    f"Ventanas temporales: "
    f"{df_contrato_periodo_v2['id_ventana'].nunique()}"
)

print(
    f"Créditos sin cobertura V2: "
    f"{len(ids_sin_cobertura_v2):,}"
)

print("-" * 65)

FINPREDICT | COHORTE ANALÍTICA V2
-----------------------------------------------------------------
Observaciones contrato-periodo: 390,661
Créditos distintos: 44,913
Ventanas temporales: 18
Créditos sin cobertura V2: 0
-----------------------------------------------------------------


In [55]:
# ============================================================
# FINPREDICT | EXPLORACIÓN PREVIA A ACTIVIDAD 6
# 18. MOROSIDAD HISTÓRICA V2 POR CONTRATO-PERIODO
# ============================================================

TOLERANCIA_MONETARIA = 0.01

cortes_v2 = sorted(
    df_contrato_periodo_v2["fecha_corte"].unique()
)

resultados_mora_v2 = []

for fecha_corte in cortes_v2:

    # --------------------------------------------------------
    # 1. Obligaciones exigibles hasta el corte
    # --------------------------------------------------------

    obligaciones = df_obligaciones.loc[
        df_obligaciones["fecha_vencimiento"] <= fecha_corte,
        [
            "id_credito",
            "fecha_vencimiento",
            "importe_exigible"
        ]
    ].copy()

    obligaciones = obligaciones.loc[
        obligaciones["id_credito"].isin(ids_cohorte)
    ]

    obligaciones = obligaciones.sort_values(
        ["id_credito", "fecha_vencimiento"]
    )

    # --------------------------------------------------------
    # 2. Pagos V2 efectuados hasta el corte
    # --------------------------------------------------------

    pagos = (
        df_movimientos_v2.loc[
            df_movimientos_v2["fecha_movimiento"] <= fecha_corte
        ]
        .groupby("id_credito")["importe"]
        .sum()
        .rename("total_pagado")
    )

    # --------------------------------------------------------
    # 3. Obligaciones acumuladas
    # --------------------------------------------------------

    obligaciones["exigible_acumulado"] = (
        obligaciones
        .groupby("id_credito")["importe_exigible"]
        .cumsum()
    )

    obligaciones = obligaciones.join(
        pagos,
        on="id_credito"
    )

    obligaciones["total_pagado"] = (
        obligaciones["total_pagado"]
        .fillna(0)
    )

    # --------------------------------------------------------
    # 4. Saldo pendiente FIFO
    # --------------------------------------------------------

    obligaciones["saldo_pendiente"] = np.clip(
        obligaciones["exigible_acumulado"]
        - obligaciones["total_pagado"],
        0,
        obligaciones["importe_exigible"]
    ).round(2)

    pendientes = obligaciones.loc[
        obligaciones["saldo_pendiente"]
        > TOLERANCIA_MONETARIA
    ]

    # --------------------------------------------------------
    # 5. Resumen por crédito
    # --------------------------------------------------------

    resumen = (
        pendientes
        .groupby("id_credito")
        .agg(
            saldo_vencido=("saldo_pendiente", "sum"),
            vencimiento_antiguo=("fecha_vencimiento", "min")
        )
        .reset_index()
    )

    resumen["fecha_corte"] = fecha_corte

    resultados_mora_v2.append(
        resumen
    )


# ------------------------------------------------------------
# 6. Consolidar estados
# ------------------------------------------------------------

df_mora_historica_v2 = pd.concat(
    resultados_mora_v2,
    ignore_index=True
)


# ------------------------------------------------------------
# 7. Integrar con la cohorte contrato-periodo
# ------------------------------------------------------------

df_estado_historico_v2 = (
    df_contrato_periodo_v2
    .merge(
        df_mora_historica_v2,
        on=[
            "id_credito",
            "fecha_corte"
        ],
        how="left",
        validate="one_to_one"
    )
)

df_estado_historico_v2["saldo_vencido"] = (
    df_estado_historico_v2[
        "saldo_vencido"
    ]
    .fillna(0)
    .round(2)
)

df_estado_historico_v2["dias_atraso"] = (
    df_estado_historico_v2["fecha_corte"]
    - df_estado_historico_v2["vencimiento_antiguo"]
).dt.days.fillna(0).astype(int)

df_estado_historico_v2["mora_90_corte"] = (
    df_estado_historico_v2["dias_atraso"] >= 90
).astype(int)


# ------------------------------------------------------------
# 8. Validaciones
# ------------------------------------------------------------

assert len(
    df_estado_historico_v2
) == len(
    df_contrato_periodo_v2
)

assert not df_estado_historico_v2.duplicated(
    ["id_credito", "fecha_corte"]
).any()

assert (
    df_estado_historico_v2["dias_atraso"] >= 0
).all()


# ------------------------------------------------------------
# 9. Resumen por ventana
# ------------------------------------------------------------

resumen_mora_historica_v2 = (
    df_estado_historico_v2
    .groupby(
        [
            "id_ventana",
            "fecha_corte",
            "uso"
        ]
    )
    .agg(
        observaciones=(
            "id_credito",
            "count"
        ),
        mora_90=(
            "mora_90_corte",
            "sum"
        ),
        tasa_mora_90=(
            "mora_90_corte",
            "mean"
        )
    )
    .reset_index()
)

resumen_mora_historica_v2[
    "tasa_mora_90_%"
] = (
    resumen_mora_historica_v2[
        "tasa_mora_90"
    ] * 100
).round(2)


# ------------------------------------------------------------
# 10. Resultados
# ------------------------------------------------------------

print("FINPREDICT | MOROSIDAD HISTÓRICA V2")
print("-" * 70)

print(
    f"Observaciones: "
    f"{len(df_estado_historico_v2):,}"
)

print(
    f"Observaciones con mora >=90: "
    f"{df_estado_historico_v2['mora_90_corte'].sum():,}"
)

display(
    resumen_mora_historica_v2[
        [
            "id_ventana",
            "fecha_corte",
            "uso",
            "observaciones",
            "mora_90",
            "tasa_mora_90_%"
        ]
    ]
)

FINPREDICT | MOROSIDAD HISTÓRICA V2
----------------------------------------------------------------------
Observaciones: 390,661
Observaciones con mora >=90: 32,071


,id_ventana,fecha_corte,uso,observaciones,mora_90,tasa_mora_90_%
0,2,2021-06-30,HISTORICO,2505,11,0.44
1,3,2021-09-30,HISTORICO,4874,78,1.60
2,4,2021-12-31,HISTORICO,7327,188,2.57
3,5,2022-03-31,HISTORICO,9649,344,3.57
4,6,2022-06-30,HISTORICO,11914,533,4.47
5,7,2022-09-30,HISTORICO,14155,749,5.29
6,8,2022-12-31,HISTORICO,16409,982,5.98
7,9,2023-03-31,HISTORICO,18608,1177,6.33
8,10,2023-06-30,HISTORICO,20785,1481,7.13
9,11,2023-09-30,HISTORICO,22812,1815,7.96


In [56]:
# ============================================================
# FINPREDICT | EXPLORACIÓN PREVIA A ACTIVIDAD 6
# 19. SEGMENTACIÓN HISTÓRICA DE LA POBLACIÓN V2
# ============================================================

df_estado_historico_v2["segmento_operativo"] = np.select(
    [
        df_estado_historico_v2["dias_atraso"] <= 60,

        df_estado_historico_v2["dias_atraso"].between(
            61,
            89,
            inclusive="both"
        ),

        df_estado_historico_v2["dias_atraso"] >= 90
    ],
    [
        "MODELO_PREVENTIVO",
        "COBRANZA_INTENSIVA",
        "MORA_90_MATERIALIZADA"
    ],
    default="REVISAR"
)

# ------------------------------------------------------------
# Resumen por ventana
# ------------------------------------------------------------

resumen_segmentacion_historica_v2 = (
    df_estado_historico_v2
    .groupby(
        [
            "id_ventana",
            "fecha_corte",
            "uso",
            "segmento_operativo"
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
    .reset_index()
)

# Garantizar columnas esperadas
for columna in [
    "MODELO_PREVENTIVO",
    "COBRANZA_INTENSIVA",
    "MORA_90_MATERIALIZADA"
]:
    if columna not in resumen_segmentacion_historica_v2.columns:
        resumen_segmentacion_historica_v2[columna] = 0

resumen_segmentacion_historica_v2["TOTAL"] = (
    resumen_segmentacion_historica_v2[
        [
            "MODELO_PREVENTIVO",
            "COBRANZA_INTENSIVA",
            "MORA_90_MATERIALIZADA"
        ]
    ]
    .sum(axis=1)
)

resumen_segmentacion_historica_v2[
    "% MODELO_PREVENTIVO"
] = (
    resumen_segmentacion_historica_v2[
        "MODELO_PREVENTIVO"
    ]
    / resumen_segmentacion_historica_v2["TOTAL"]
    * 100
).round(2)

resumen_segmentacion_historica_v2[
    "% COBRANZA_INTENSIVA"
] = (
    resumen_segmentacion_historica_v2[
        "COBRANZA_INTENSIVA"
    ]
    / resumen_segmentacion_historica_v2["TOTAL"]
    * 100
).round(2)

resumen_segmentacion_historica_v2[
    "% MORA_90"
] = (
    resumen_segmentacion_historica_v2[
        "MORA_90_MATERIALIZADA"
    ]
    / resumen_segmentacion_historica_v2["TOTAL"]
    * 100
).round(2)

# ------------------------------------------------------------
# Población histórica que utilizará FinPredict
# ------------------------------------------------------------

df_historico_preventivo_v2 = (
    df_estado_historico_v2.loc[
        df_estado_historico_v2[
            "segmento_operativo"
        ] == "MODELO_PREVENTIVO"
    ]
    .copy()
)

# ------------------------------------------------------------
# Validaciones
# ------------------------------------------------------------

assert (
    resumen_segmentacion_historica_v2["TOTAL"].sum()
    == len(df_estado_historico_v2)
)

assert (
    df_historico_preventivo_v2["dias_atraso"]
    <= 60
).all()

assert not df_historico_preventivo_v2.duplicated(
    ["id_credito", "fecha_corte"]
).any()

# ------------------------------------------------------------
# Resultados
# ------------------------------------------------------------

print("FINPREDICT | SEGMENTACIÓN HISTÓRICA V2")
print("-" * 75)

print(
    f"Observaciones totales: "
    f"{len(df_estado_historico_v2):,}"
)

print(
    f"Observaciones modelo preventivo: "
    f"{len(df_historico_preventivo_v2):,}"
)

print(
    f"Cobertura histórica del modelo: "
    f"{len(df_historico_preventivo_v2) / len(df_estado_historico_v2):.2%}"
)

print("\nDistribución por ventana:")

display(
    resumen_segmentacion_historica_v2[
        [
            "id_ventana",
            "fecha_corte",
            "uso",
            "MODELO_PREVENTIVO",
            "COBRANZA_INTENSIVA",
            "MORA_90_MATERIALIZADA",
            "% MODELO_PREVENTIVO"
        ]
    ]
)

FINPREDICT | SEGMENTACIÓN HISTÓRICA V2
---------------------------------------------------------------------------
Observaciones totales: 390,661
Observaciones modelo preventivo: 336,837
Cobertura histórica del modelo: 86.22%

Distribución por ventana:


segmento_operativo,id_ventana,fecha_corte,uso,MODELO_PREVENTIVO,COBRANZA_INTENSIVA,MORA_90_MATERIALIZADA,% MODELO_PREVENTIVO
0,2,2021-06-30,HISTORICO,2440,54,11,97.41
1,3,2021-09-30,HISTORICO,4632,164,78,95.03
2,4,2021-12-31,HISTORICO,6794,345,188,92.73
3,5,2022-03-31,HISTORICO,8842,463,344,91.64
4,6,2022-06-30,HISTORICO,10796,585,533,90.62
5,7,2022-09-30,HISTORICO,12634,772,749,89.25
6,8,2022-12-31,HISTORICO,14575,852,982,88.82
7,9,2023-03-31,HISTORICO,16411,1020,1177,88.19
8,10,2023-06-30,HISTORICO,18074,1230,1481,86.96
9,11,2023-09-30,HISTORICO,19681,1316,1815,86.27


In [57]:
# ============================================================
# FINPREDICT | EXPLORACIÓN PREVIA A ACTIVIDAD 6
# 20. CONSTRUCCIÓN DEL TARGET HISTÓRICO V2
# ============================================================

TOLERANCIA = TOLERANCIA_MONETARIA

# ------------------------------------------------------------
# 1. Base preventiva por contrato-periodo
# ------------------------------------------------------------

df_base_target_v2 = (
    df_historico_preventivo_v2[
        [
            "id_credito",
            "id_ventana",
            "fecha_corte",
            "inicio_prediccion",
            "fin_prediccion",
            "uso"
        ]
    ]
    .copy()
)

# ------------------------------------------------------------
# 2. Preparar pagos acumulados V2
# ------------------------------------------------------------

pagos_ordenados_v2 = (
    df_movimientos_v2[
        [
            "id_credito",
            "fecha_movimiento",
            "importe"
        ]
    ]
    .sort_values(
        ["id_credito", "fecha_movimiento"]
    )
    .copy()
)

pagos_ordenados_v2["pagado_acumulado"] = (
    pagos_ordenados_v2
    .groupby("id_credito")["importe"]
    .cumsum()
)

# ------------------------------------------------------------
# 3. Preparar obligaciones acumuladas
# ------------------------------------------------------------

obligaciones_ordenadas_v2 = (
    df_obligaciones[
        [
            "id_credito",
            "fecha_vencimiento",
            "importe_exigible"
        ]
    ]
    .loc[
        lambda x: x["id_credito"].isin(ids_cohorte)
    ]
    .sort_values(
        ["id_credito", "fecha_vencimiento"]
    )
    .copy()
)

obligaciones_ordenadas_v2["exigible_acumulado"] = (
    obligaciones_ordenadas_v2
    .groupby("id_credito")["importe_exigible"]
    .cumsum()
)

# ------------------------------------------------------------
# 4. Evaluar cada ventana
# ------------------------------------------------------------

resultados_target_v2 = []

for ventana in df_ventanas_base.itertuples(index=False):

    poblacion = df_base_target_v2.loc[
        df_base_target_v2["id_ventana"] == ventana.id_ventana,
        "id_credito"
    ]

    if poblacion.empty:
        continue

    # Obligaciones capaces de alcanzar 90 días
    # dentro de la ventana futura
    obligaciones_evaluar = (
        obligaciones_ordenadas_v2.loc[
            obligaciones_ordenadas_v2["id_credito"].isin(poblacion)
            & (
                obligaciones_ordenadas_v2["fecha_vencimiento"]
                <= ventana.fin_prediccion - pd.Timedelta(days=90)
            )
        ]
        .copy()
    )

    obligaciones_evaluar["fecha_revision"] = (
        obligaciones_evaluar["fecha_vencimiento"]
        + pd.Timedelta(days=90)
    )

    # Si la obligación ya tenía suficiente antigüedad,
    # se evalúa desde el inicio del trimestre
    obligaciones_evaluar["fecha_revision"] = (
        obligaciones_evaluar["fecha_revision"]
        .clip(lower=ventana.inicio_prediccion)
    )

    obligaciones_evaluar = obligaciones_evaluar.loc[
        obligaciones_evaluar["fecha_revision"]
        <= ventana.fin_prediccion
    ].copy()

    # Pagos de la población correspondiente
    pagos_ventana = (
        pagos_ordenados_v2.loc[
            pagos_ordenados_v2["id_credito"].isin(poblacion),
            [
                "id_credito",
                "fecha_movimiento",
                "pagado_acumulado"
            ]
        ]
        .copy()
    )

    # Orden requerido por merge_asof
    obligaciones_evaluar = (
        obligaciones_evaluar
        .sort_values(
            ["fecha_revision", "id_credito"]
        )
    )

    pagos_ventana = (
        pagos_ventana
        .sort_values(
            ["fecha_movimiento", "id_credito"]
        )
    )

    # --------------------------------------------------------
    # Conciliación pagos vs fecha de revisión
    # --------------------------------------------------------

    conciliacion = pd.merge_asof(
        obligaciones_evaluar,
        pagos_ventana,
        left_on="fecha_revision",
        right_on="fecha_movimiento",
        by="id_credito",
        direction="backward",
        allow_exact_matches=True
    )

    conciliacion["pagado_acumulado"] = (
        conciliacion["pagado_acumulado"]
        .fillna(0)
    )

    conciliacion["saldo_90_pendiente"] = (
        conciliacion["exigible_acumulado"]
        - conciliacion["pagado_acumulado"]
    ).clip(lower=0)

    conciliacion["evento_90"] = (
        conciliacion["saldo_90_pendiente"]
        > TOLERANCIA
    ).astype(int)

    # --------------------------------------------------------
    # Una etiqueta por crédito y ventana
    # --------------------------------------------------------

    etiquetas = (
        conciliacion
        .groupby("id_credito")["evento_90"]
        .max()
        .rename("incumplimiento_90d")
        .reset_index()
    )

    etiquetas["id_ventana"] = ventana.id_ventana

    resultados_target_v2.append(
        etiquetas
    )

# ------------------------------------------------------------
# 5. Consolidar
# ------------------------------------------------------------

df_target_historico_v2 = pd.concat(
    resultados_target_v2,
    ignore_index=True
)

df_target_historico_v2 = (
    df_base_target_v2
    .merge(
        df_target_historico_v2,
        on=[
            "id_credito",
            "id_ventana"
        ],
        how="left",
        validate="one_to_one"
    )
)

df_target_historico_v2["incumplimiento_90d"] = (
    df_target_historico_v2[
        "incumplimiento_90d"
    ]
    .fillna(0)
    .astype(int)
)

# ------------------------------------------------------------
# 6. Validaciones
# ------------------------------------------------------------

assert len(
    df_target_historico_v2
) == len(
    df_base_target_v2
)

assert not df_target_historico_v2.duplicated(
    [
        "id_credito",
        "id_ventana"
    ]
).any()

assert df_target_historico_v2[
    "incumplimiento_90d"
].isin([0, 1]).all()

# ------------------------------------------------------------
# 7. Resumen por trimestre
# ------------------------------------------------------------

resumen_target_v2 = (
    df_target_historico_v2
    .groupby(
        [
            "id_ventana",
            "fecha_corte",
            "uso"
        ]
    )
    .agg(
        observaciones=(
            "id_credito",
            "count"
        ),
        incumplimientos=(
            "incumplimiento_90d",
            "sum"
        ),
        tasa_incumplimiento=(
            "incumplimiento_90d",
            "mean"
        )
    )
    .reset_index()
)

resumen_target_v2[
    "tasa_incumplimiento_%"
] = (
    resumen_target_v2[
        "tasa_incumplimiento"
    ] * 100
).round(2)

# ------------------------------------------------------------
# 8. Resultados
# ------------------------------------------------------------

print("FINPREDICT | TARGET HISTÓRICO V2")
print("-" * 75)

print(
    f"Observaciones etiquetadas: "
    f"{len(df_target_historico_v2):,}"
)

print(
    f"Incumplimientos totales: "
    f"{df_target_historico_v2['incumplimiento_90d'].sum():,}"
)

print(
    f"Prevalencia global: "
    f"{df_target_historico_v2['incumplimiento_90d'].mean():.2%}"
)

print("\nDistribución por ventana:")

display(
    resumen_target_v2[
        [
            "id_ventana",
            "fecha_corte",
            "uso",
            "observaciones",
            "incumplimientos",
            "tasa_incumplimiento_%"
        ]
    ]
)

FINPREDICT | TARGET HISTÓRICO V2
---------------------------------------------------------------------------
Observaciones etiquetadas: 336,837
Incumplimientos totales: 12,898
Prevalencia global: 3.83%

Distribución por ventana:


,id_ventana,fecha_corte,uso,observaciones,incumplimientos,tasa_incumplimiento_%
0,2,2021-06-30,HISTORICO,2440,91,3.73
1,3,2021-09-30,HISTORICO,4632,188,4.06
2,4,2021-12-31,HISTORICO,6794,237,3.49
3,5,2022-03-31,HISTORICO,8842,329,3.72
4,6,2022-06-30,HISTORICO,10796,421,3.90
5,7,2022-09-30,HISTORICO,12634,481,3.81
6,8,2022-12-31,HISTORICO,14575,532,3.65
7,9,2023-03-31,HISTORICO,16411,671,4.09
8,10,2023-06-30,HISTORICO,18074,764,4.23
9,11,2023-09-30,HISTORICO,19681,696,3.54


In [58]:
# ============================================================
# FINPREDICT | EXPLORACIÓN PREVIA A ACTIVIDAD 6
# 21. CONSTRUCCIÓN DE LA BASE ANALÍTICA V2
# ============================================================

# ------------------------------------------------------------
# 1. Variables contractuales estáticas
# ------------------------------------------------------------

variables_contractuales = [
    "id_credito",
    "fecha_apertura",
    "monto_original",
    "plazo_meses",
    "tasa_interes_anual",
    "pago_mensual",
    "ingreso_mensual",
    "tipo_empleo",
    "edad",
    "region"
]

df_variables_contrato_v2 = (
    df_contratos[
        variables_contractuales
    ]
    .copy()
)

# ------------------------------------------------------------
# 2. Variables de estado V2
# ------------------------------------------------------------

df_variables_estado_v2 = (
    df_estado_historico_v2[
        [
            "id_credito",
            "fecha_corte",
            "saldo_vencido",
            "dias_atraso",
            "mora_90_corte"
        ]
    ]
    .copy()
)

# ------------------------------------------------------------
# 3. Integrar target + estado + contrato
# ------------------------------------------------------------

df_analitico_v2 = (
    df_target_historico_v2
    .merge(
        df_variables_estado_v2,
        on=[
            "id_credito",
            "fecha_corte"
        ],
        how="left",
        validate="one_to_one"
    )
    .merge(
        df_variables_contrato_v2,
        on="id_credito",
        how="left",
        validate="many_to_one"
    )
)

# ------------------------------------------------------------
# 4. Antigüedad al corte
# ------------------------------------------------------------

df_analitico_v2["antiguedad_meses"] = (
    (
        df_analitico_v2["fecha_corte"].dt.year
        - df_analitico_v2["fecha_apertura"].dt.year
    ) * 12
    +
    (
        df_analitico_v2["fecha_corte"].dt.month
        - df_analitico_v2["fecha_apertura"].dt.month
    )
)

# ------------------------------------------------------------
# 5. Indicadores financieros
# ------------------------------------------------------------

df_analitico_v2["relacion_pago_ingreso"] = (
    df_analitico_v2["pago_mensual"]
    / df_analitico_v2["ingreso_mensual"]
)

df_analitico_v2["saldo_vencido_mensualidades"] = (
    df_analitico_v2["saldo_vencido"]
    / df_analitico_v2["pago_mensual"]
)

# ------------------------------------------------------------
# 6. Validaciones
# ------------------------------------------------------------

assert len(
    df_analitico_v2
) == len(
    df_target_historico_v2
)

assert not df_analitico_v2.duplicated(
    [
        "id_credito",
        "fecha_corte"
    ]
).any()

assert (
    df_analitico_v2["dias_atraso"] <= 60
).all()

assert (
    df_analitico_v2["mora_90_corte"] == 0
).all()

assert df_analitico_v2[
    "incumplimiento_90d"
].isin([0, 1]).all()

# ------------------------------------------------------------
# 7. Resumen
# ------------------------------------------------------------

print("FINPREDICT | BASE ANALÍTICA V2")
print("-" * 70)

print(
    f"Observaciones: "
    f"{len(df_analitico_v2):,}"
)

print(
    f"Créditos distintos: "
    f"{df_analitico_v2['id_credito'].nunique():,}"
)

print(
    f"Variables actuales: "
    f"{df_analitico_v2.shape[1]}"
)

print(
    f"Incumplimientos: "
    f"{df_analitico_v2['incumplimiento_90d'].sum():,}"
)

print(
    f"Prevalencia: "
    f"{df_analitico_v2['incumplimiento_90d'].mean():.2%}"
)

print("-" * 70)

display(
    df_analitico_v2.head()
)

FINPREDICT | BASE ANALÍTICA V2
----------------------------------------------------------------------
Observaciones: 336,837
Créditos distintos: 44,445
Variables actuales: 22
Incumplimientos: 12,898
Prevalencia: 3.83%
----------------------------------------------------------------------


,id_credito,id_ventana,fecha_corte,inicio_prediccion,fin_prediccion,uso,incumplimiento_90d,saldo_vencido,dias_atraso,mora_90_corte,...,plazo_meses,tasa_interes_anual,pago_mensual,ingreso_mensual,tipo_empleo,edad,region,antiguedad_meses,relacion_pago_ingreso,saldo_vencido_mensualidades
0,HIP000056,2,2021-06-30,2021-07-01,2021-09-30,HISTORICO,0,18512.38,9,0,...,240,7.2769,37404.45,126056.44,ASALARIADO,61,CENTRO,3,0.296728,0.494925
1,HIP000104,2,2021-06-30,2021-07-01,2021-09-30,HISTORICO,0,2241.39,16,0,...,180,9.4087,13777.62,39595.59,ASALARIADO,42,SUR,5,0.347958,0.162683
2,HIP000117,2,2021-06-30,2021-07-01,2021-09-30,HISTORICO,0,0.00,0,0,...,240,12.5357,22478.21,60683.97,ASALARIADO,58,CENTRO,3,0.370414,0.000000
3,HIP000170,2,2021-06-30,2021-07-01,2021-09-30,HISTORICO,0,3104.72,19,0,...,240,8.0479,12965.36,49894.86,INDEPENDIENTE,52,NORTE,4,0.259854,0.239463
4,HIP000233,2,2021-06-30,2021-07-01,2021-09-30,HISTORICO,0,0.00,0,0,...,240,9.0836,26399.53,79239.25,ASALARIADO,34,NORTE,4,0.333162,0.000000


In [61]:
# ============================================================
# FINPREDICT | EXPLORACIÓN PREVIA A ACTIVIDAD 6
# 22.1 DIAGNÓSTICO DE COLUMNAS DE MOVIMIENTOS V2
# ============================================================

print("Columnas de df_movimientos_v2:")
print(df_movimientos_v2.columns.tolist())

print("\nPrimeros registros:")
display(df_movimientos_v2.head())

print("\nTipos de datos:")
display(df_movimientos_v2.dtypes)

Columnas de df_movimientos_v2:
['id_credito', 'numero_pago_origen', 'fecha_movimiento', 'tipo_movimiento', 'importe', 'resultado_simulado']

Primeros registros:


,id_credito,numero_pago_origen,fecha_movimiento,tipo_movimiento,importe,resultado_simulado
0,HIP000001,1,2021-07-17,SPEI,43198.32,COMPLETO
1,HIP000001,2,2021-08-17,SPEI,43198.32,COMPLETO
2,HIP000001,3,2021-09-22,SPEI,43198.32,COMPLETO
3,HIP000001,4,2021-10-09,CAJA,43198.32,COMPLETO
4,HIP000001,5,2021-11-18,SPEI,43198.32,COMPLETO



Tipos de datos:


,0
id_credito,object
numero_pago_origen,object
fecha_movimiento,datetime64[ns]
tipo_movimiento,object
importe,float64
resultado_simulado,object


In [62]:
# ============================================================
# FINPREDICT | EXPLORACIÓN PREVIA A ACTIVIDAD 6
# 22. VARIABLES TRANSACCIONALES V2 - ÚLTIMOS 3 MESES
# ============================================================

resultados_3m_v2 = []

for ventana in df_ventanas_base.itertuples(index=False):

    fecha_corte = ventana.fecha_corte
    fecha_inicio_3m = fecha_corte - pd.DateOffset(months=3)

    poblacion = df_analitico_v2.loc[
        df_analitico_v2["id_ventana"] == ventana.id_ventana,
        "id_credito"
    ]

    if poblacion.empty:
        continue

    # --------------------------------------------------------
    # 1. Movimientos de los 3 meses previos al corte
    # --------------------------------------------------------

    movimientos_3m = df_movimientos_v2.loc[
        df_movimientos_v2["id_credito"].isin(poblacion)
        & (
            df_movimientos_v2["fecha_movimiento"]
            > fecha_inicio_3m
        )
        & (
            df_movimientos_v2["fecha_movimiento"]
            <= fecha_corte
        )
    ].copy()

    resumen_movimientos = (
        movimientos_3m
        .groupby("id_credito")
        .agg(
            importe_pagado_3m=(
                "importe",
                "sum"
            ),
            numero_pagos_3m=(
                "importe",
                "size"
            ),
            pagos_parciales_3m=(
                "resultado_simulado",
                lambda x: (x == "PARCIAL").sum()
            ),
            regularizaciones_3m=(
                "resultado_simulado",
                lambda x: (x == "REGULARIZACION").sum()
            )
        )
        .reset_index()
    )

    # --------------------------------------------------------
    # 2. Obligaciones exigibles en los 3 meses previos
    # --------------------------------------------------------

    obligaciones_3m = df_obligaciones.loc[
        df_obligaciones["id_credito"].isin(poblacion)
        & (
            df_obligaciones["fecha_vencimiento"]
            > fecha_inicio_3m
        )
        & (
            df_obligaciones["fecha_vencimiento"]
            <= fecha_corte
        )
    ].copy()

    resumen_obligaciones = (
        obligaciones_3m
        .groupby("id_credito")
        .agg(
            importe_exigible_3m=(
                "importe_exigible",
                "sum"
            ),
            obligaciones_3m=(
                "importe_exigible",
                "size"
            )
        )
        .reset_index()
    )

    # --------------------------------------------------------
    # 3. Consolidar por crédito
    # --------------------------------------------------------

    base_ventana = (
        pd.DataFrame({
            "id_credito": poblacion.values
        })
        .merge(
            resumen_movimientos,
            on="id_credito",
            how="left",
            validate="one_to_one"
        )
        .merge(
            resumen_obligaciones,
            on="id_credito",
            how="left",
            validate="one_to_one"
        )
    )

    base_ventana["id_ventana"] = ventana.id_ventana

    resultados_3m_v2.append(base_ventana)

# ------------------------------------------------------------
# 4. Consolidar todas las ventanas
# ------------------------------------------------------------

df_variables_3m_v2 = pd.concat(
    resultados_3m_v2,
    ignore_index=True
)

columnas_rellenar = [
    "importe_pagado_3m",
    "numero_pagos_3m",
    "pagos_parciales_3m",
    "regularizaciones_3m",
    "importe_exigible_3m",
    "obligaciones_3m"
]

df_variables_3m_v2[columnas_rellenar] = (
    df_variables_3m_v2[columnas_rellenar]
    .fillna(0)
)

# ------------------------------------------------------------
# 5. Porcentaje de pago
# ------------------------------------------------------------

df_variables_3m_v2["porcentaje_pago_3m"] = np.where(
    df_variables_3m_v2["importe_exigible_3m"] > 0,
    (
        df_variables_3m_v2["importe_pagado_3m"]
        / df_variables_3m_v2["importe_exigible_3m"]
    ),
    0
)

# ------------------------------------------------------------
# 6. Integrar con base analítica
# ------------------------------------------------------------

df_analitico_v2 = (
    df_analitico_v2
    .merge(
        df_variables_3m_v2,
        on=[
            "id_credito",
            "id_ventana"
        ],
        how="left",
        validate="one_to_one"
    )
)

# ------------------------------------------------------------
# 7. Categoría descriptiva de cobertura
# ------------------------------------------------------------

df_analitico_v2["categoria_cobertura_3m"] = pd.cut(
    df_analitico_v2["porcentaje_pago_3m"],
    bins=[
        -np.inf,
        0.50,
        0.80,
        1.00,
        np.inf
    ],
    labels=[
        "BAJA",
        "MEDIA",
        "ALTA",
        "SOBRECUMPLIMIENTO"
    ],
    right=False
)

# ------------------------------------------------------------
# 8. Validaciones
# ------------------------------------------------------------

assert len(df_analitico_v2) == 336_837

assert not df_analitico_v2.duplicated(
    [
        "id_credito",
        "fecha_corte"
    ]
).any()

assert df_analitico_v2[
    columnas_rellenar
].notna().all().all()

assert np.isfinite(
    df_analitico_v2[
        ["porcentaje_pago_3m"]
    ].to_numpy()
).all()

# ------------------------------------------------------------
# 9. Resumen
# ------------------------------------------------------------

print("FINPREDICT | VARIABLES TRANSACCIONALES V2")
print("-" * 70)

print(
    f"Observaciones: "
    f"{len(df_analitico_v2):,}"
)

print(
    f"Variables totales: "
    f"{df_analitico_v2.shape[1]}"
)

print("\nPromedios por clase:")

display(
    df_analitico_v2
    .groupby("incumplimiento_90d")[
        [
            "importe_pagado_3m",
            "numero_pagos_3m",
            "importe_exigible_3m",
            "obligaciones_3m",
            "porcentaje_pago_3m",
            "pagos_parciales_3m",
            "regularizaciones_3m"
        ]
    ]
    .mean()
    .round(4)
)

FINPREDICT | VARIABLES TRANSACCIONALES V2
----------------------------------------------------------------------
Observaciones: 336,837
Variables totales: 30

Promedios por clase:


,importe_pagado_3m,numero_pagos_3m,importe_exigible_3m,obligaciones_3m,porcentaje_pago_3m,pagos_parciales_3m,regularizaciones_3m
incumplimiento_90d,,,,,,,
0,87657.2024,3.4555,86074.4260,3.0048,1.0187,0.2874,0.5443
1,84559.5259,3.0906,86087.0847,3.0017,0.9816,0.7688,0.4504


In [63]:
# ============================================================
# FINPREDICT | EXPLORACIÓN PREVIA A ACTIVIDAD 6
# 23. CREACIÓN DE DATASETS FINALES V2
# ============================================================

# ------------------------------------------------------------
# 1. Orden final de columnas
# ------------------------------------------------------------

columnas_finales_v2 = [
    # Control
    "id_credito",
    "id_ventana",
    "fecha_corte",
    "inicio_prediccion",
    "fin_prediccion",
    "uso",
    "fecha_apertura",

    # Target
    "incumplimiento_90d",

    # Contractuales
    "monto_original",
    "plazo_meses",
    "tasa_interes_anual",
    "pago_mensual",
    "ingreso_mensual",
    "antiguedad_meses",

    # Morosidad
    "saldo_vencido",
    "dias_atraso",

    # Indicadores financieros
    "relacion_pago_ingreso",
    "saldo_vencido_mensualidades",

    # Comportamiento 3 meses
    "importe_pagado_3m",
    "numero_pagos_3m",
    "importe_exigible_3m",
    "obligaciones_3m",
    "porcentaje_pago_3m",

    # Auditoría
    "mora_90_corte",
    "tipo_empleo",
    "edad",
    "region",
    "pagos_parciales_3m",
    "regularizaciones_3m",
    "categoria_cobertura_3m"
]

df_preparado_v2 = (
    df_analitico_v2[
        columnas_finales_v2
    ]
    .copy()
)

# ------------------------------------------------------------
# 2. Separación temporal
# ------------------------------------------------------------

df_desarrollo_v2 = (
    df_preparado_v2.loc[
        df_preparado_v2["uso"] == "HISTORICO"
    ]
    .copy()
)

df_evaluacion_v2 = (
    df_preparado_v2.loc[
        df_preparado_v2["uso"] == "EVALUACION_FINAL"
    ]
    .copy()
)

# ------------------------------------------------------------
# 3. Validaciones estructurales
# ------------------------------------------------------------

assert len(df_preparado_v2) == 336_837

assert (
    len(df_desarrollo_v2)
    + len(df_evaluacion_v2)
    == len(df_preparado_v2)
)

assert not df_preparado_v2.duplicated(
    ["id_credito", "fecha_corte"]
).any()

assert df_preparado_v2[
    "incumplimiento_90d"
].isin([0, 1]).all()

assert (
    df_preparado_v2["dias_atraso"] <= 60
).all()

assert (
    df_preparado_v2["mora_90_corte"] == 0
).all()

# ------------------------------------------------------------
# 4. Resumen
# ------------------------------------------------------------

print("FINPREDICT | DATASETS FINALES V2")
print("-" * 70)

print(
    f"Preparado V2: "
    f"{len(df_preparado_v2):,} observaciones"
)

print(
    f"Desarrollo V2: "
    f"{len(df_desarrollo_v2):,} observaciones"
)

print(
    f"Evaluación V2: "
    f"{len(df_evaluacion_v2):,} observaciones"
)

print(
    f"Variables: "
    f"{df_preparado_v2.shape[1]}"
)

print("\nDistribución de clases:")

display(
    pd.DataFrame({
        "dataset": [
            "Desarrollo",
            "Evaluación final"
        ],
        "observaciones": [
            len(df_desarrollo_v2),
            len(df_evaluacion_v2)
        ],
        "incumplimientos": [
            df_desarrollo_v2[
                "incumplimiento_90d"
            ].sum(),
            df_evaluacion_v2[
                "incumplimiento_90d"
            ].sum()
        ],
        "prevalencia_%": [
            (
                df_desarrollo_v2[
                    "incumplimiento_90d"
                ].mean() * 100
            ),
            (
                df_evaluacion_v2[
                    "incumplimiento_90d"
                ].mean() * 100
            )
        ]
    }).round(2)
)

FINPREDICT | DATASETS FINALES V2
----------------------------------------------------------------------
Preparado V2: 336,837 observaciones
Desarrollo V2: 303,234 observaciones
Evaluación V2: 33,603 observaciones
Variables: 30

Distribución de clases:


,dataset,observaciones,incumplimientos,prevalencia_%
0,Desarrollo,303234,11642,3.84
1,Evaluación final,33603,1256,3.74


In [64]:
# ============================================================
# FINPREDICT | EXPLORACIÓN PREVIA A ACTIVIDAD 6
# 24. GUARDADO OFICIAL DE DATASETS V2
# ============================================================

import json

# ------------------------------------------------------------
# 1. Rutas de salida
# ------------------------------------------------------------

ruta_preparado_v2 = (
    RUTA_DATOS_LIMP
    / "finpredict_preparado_v2.parquet"
)

ruta_desarrollo_v2 = (
    RUTA_DATOS_LIMP
    / "finpredict_desarrollo_v2.parquet"
)

ruta_evaluacion_v2 = (
    RUTA_DATOS_LIMP
    / "finpredict_evaluacion_v2.parquet"
)

ruta_manifiesto_v2 = (
    RUTA_DATOS_LIMP
    / "manifiesto_v2.json"
)

# ------------------------------------------------------------
# 2. Guardar datasets
# ------------------------------------------------------------

df_preparado_v2.to_parquet(
    ruta_preparado_v2,
    index=False,
    compression="snappy"
)

df_desarrollo_v2.to_parquet(
    ruta_desarrollo_v2,
    index=False,
    compression="snappy"
)

df_evaluacion_v2.to_parquet(
    ruta_evaluacion_v2,
    index=False,
    compression="snappy"
)

# ------------------------------------------------------------
# 3. Manifiesto final V2
# ------------------------------------------------------------

manifiesto_final_v2 = {

    "version": "v2",

    "proyecto": "FinPredict",

    "proposito": (
        "Dataset analítico para evaluación y validación "
        "de modelos de riesgo de incumplimiento hipotecario."
    ),

    "origen": {
        "contratos_iniciales": int(len(df_contratos)),
        "contratos_simulables_v2": int(len(ids_simulables_v2)),
        "cohorte_historica_creditos": int(
            df_contrato_periodo_v2["id_credito"].nunique()
        ),
        "ventanas_temporales": int(
            df_contrato_periodo_v2["id_ventana"].nunique()
        )
    },

    "regla_negocio": {
        "modelo_preventivo": "0 a 60 días de atraso",
        "cobranza_intensiva": "61 a 89 días de atraso",
        "mora_materializada": "90 días o más"
    },

    "dataset_preparado": {
        "observaciones": int(len(df_preparado_v2)),
        "creditos_distintos": int(
            df_preparado_v2["id_credito"].nunique()
        ),
        "variables": int(df_preparado_v2.shape[1]),
        "incumplimientos": int(
            df_preparado_v2["incumplimiento_90d"].sum()
        ),
        "prevalencia": float(
            df_preparado_v2["incumplimiento_90d"].mean()
        )
    },

    "dataset_desarrollo": {
        "observaciones": int(len(df_desarrollo_v2)),
        "incumplimientos": int(
            df_desarrollo_v2["incumplimiento_90d"].sum()
        ),
        "prevalencia": float(
            df_desarrollo_v2["incumplimiento_90d"].mean()
        )
    },

    "dataset_evaluacion": {
        "observaciones": int(len(df_evaluacion_v2)),
        "incumplimientos": int(
            df_evaluacion_v2["incumplimiento_90d"].sum()
        ),
        "prevalencia": float(
            df_evaluacion_v2["incumplimiento_90d"].mean()
        ),
        "fecha_corte": "2025-09-30",
        "inicio_prediccion": "2025-10-01",
        "fin_prediccion": "2025-12-31"
    },

    "target": {
        "nombre": "incumplimiento_90d",
        "definicion": (
            "Saldo materialmente pendiente cuando una obligación "
            "alcanza 90 días de atraso dentro de la ventana futura."
        )
    },

    "criterios_v2": {
        "motivo": (
            "Reducir el comportamiento excesivamente determinista "
            "observado en V1, manteniendo coherencia temporal y "
            "desbalance natural."
        ),
        "prevalencia_global": float(
            df_preparado_v2["incumplimiento_90d"].mean()
        ),
        "seed": int(SEED)
    }
}

with open(
    ruta_manifiesto_v2,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifiesto_final_v2,
        f,
        ensure_ascii=False,
        indent=4
    )

# ------------------------------------------------------------
# 4. Verificación de lectura
# ------------------------------------------------------------

check_preparado = pd.read_parquet(
    ruta_preparado_v2
)

check_desarrollo = pd.read_parquet(
    ruta_desarrollo_v2
)

check_evaluacion = pd.read_parquet(
    ruta_evaluacion_v2
)

assert len(check_preparado) == len(df_preparado_v2)
assert len(check_desarrollo) == len(df_desarrollo_v2)
assert len(check_evaluacion) == len(df_evaluacion_v2)

assert list(check_preparado.columns) == list(
    df_preparado_v2.columns
)

# ------------------------------------------------------------
# 5. Resultado
# ------------------------------------------------------------

print("FINPREDICT | V2 GUARDADA OFICIALMENTE")
print("-" * 70)

print(f"✓ {ruta_preparado_v2.name}")
print(f"✓ {ruta_desarrollo_v2.name}")
print(f"✓ {ruta_evaluacion_v2.name}")
print(f"✓ {ruta_manifiesto_v2.name}")

print("-" * 70)

print(
    f"Preparado: {len(check_preparado):,}"
)

print(
    f"Desarrollo: {len(check_desarrollo):,}"
)

print(
    f"Evaluación: {len(check_evaluacion):,}"
)

print("-" * 70)
print("Integridad de archivos verificada correctamente.")

FINPREDICT | V2 GUARDADA OFICIALMENTE
----------------------------------------------------------------------
✓ finpredict_preparado_v2.parquet
✓ finpredict_desarrollo_v2.parquet
✓ finpredict_evaluacion_v2.parquet
✓ manifiesto_v2.json
----------------------------------------------------------------------
Preparado: 336,837
Desarrollo: 303,234
Evaluación: 33,603
----------------------------------------------------------------------
Integridad de archivos verificada correctamente.


In [65]:
# ============================================================
# FINPREDICT | RESPALDO V2 EN GOOGLE DRIVE
# ============================================================

from google.colab import drive
from pathlib import Path
import shutil

# 1. Montar Google Drive
drive.mount("/content/drive")

# 2. Crear carpeta de respaldo
RUTA_BACKUP_V2 = Path(
    "/content/drive/MyDrive/FinPredict-Actividad6/datos_v2"
)

RUTA_BACKUP_V2.mkdir(
    parents=True,
    exist_ok=True
)

# 3. Archivos que queremos conservar
archivos_respaldo = {
    # Datasets finales
    RUTA_DATOS_LIMP / "finpredict_preparado_v2.parquet":
        RUTA_BACKUP_V2 / "finpredict_preparado_v2.parquet",

    RUTA_DATOS_LIMP / "finpredict_desarrollo_v2.parquet":
        RUTA_BACKUP_V2 / "finpredict_desarrollo_v2.parquet",

    RUTA_DATOS_LIMP / "finpredict_evaluacion_v2.parquet":
        RUTA_BACKUP_V2 / "finpredict_evaluacion_v2.parquet",

    RUTA_DATOS_LIMP / "manifiesto_v2.json":
        RUTA_BACKUP_V2 / "manifiesto_v2.json",

    # Checkpoint pesado de simulación
    RUTA_V2 / "movimientos_v2.parquet":
        RUTA_BACKUP_V2 / "movimientos_v2.parquet",

    RUTA_V2 / "perfiles_v2.parquet":
        RUTA_BACKUP_V2 / "perfiles_v2.parquet",

    RUTA_V2 / "validacion_target_v2.parquet":
        RUTA_BACKUP_V2 / "validacion_target_v2.parquet"
}

# 4. Copiar y verificar
for origen, destino in archivos_respaldo.items():

    assert origen.exists(), f"No existe: {origen}"

    shutil.copy2(
        origen,
        destino
    )

    assert destino.exists(), f"No se copió: {destino}"

    print(f"✓ {destino.name}")

print("-" * 70)
print(f"Respaldo completo en:")
print(RUTA_BACKUP_V2)

Mounted at /content/drive
✓ finpredict_preparado_v2.parquet
✓ finpredict_desarrollo_v2.parquet
✓ finpredict_evaluacion_v2.parquet
✓ manifiesto_v2.json
✓ movimientos_v2.parquet
✓ perfiles_v2.parquet
✓ validacion_target_v2.parquet
----------------------------------------------------------------------
Respaldo completo en:
/content/drive/MyDrive/FinPredict-Actividad6/datos_v2
